### Creates `openalex.works.superlocations` in Walden End to End workflow

In [ ]:
-- oxjob #2102 (policy: plans/retractions.md § 5: a withdrawn preprint is pulled, is_retracted stays strict, the record
-- stays): arXiv papers whose CURRENT version is a withdrawal. arXiv registers every paper with DataCite
-- (10.48550/arXiv.<id>, client arxiv.content) and, when the authors or arXiv withdraw it, adds a date
-- {dateType 'Withdrawn', dateInformation 'v<N>; <comment>'} at attributes.version N. A later real version keeps the old
-- date on an older v (2510.13596: withdrawn at v2, replaced by v3), so only a withdrawal at the record's current version
-- counts. Agreed with arXiv's own arXivRaw record on 260 of 260 sampled papers (oxjob #2102 EXPLORE.md); the oai_dc comment
-- text misses withdrawals worded otherwise. 17,005 papers on 17 Nov 2026 (17,145 works). Read below by repo_superlocations
-- (no PDF link: the unversioned https://arxiv.org/pdf/<id> answers 404 'withdrawn' for them) and by CreateWorksBase
-- (is_pulled, on the work's own DOI only). Revert: git revert the commit (both readers go with it), then DROP TABLE
-- openalex.works.arxiv_withdrawals; never drop it while a reader is live.
CREATE OR REPLACE TABLE openalex.works.arxiv_withdrawals
COMMENT 'oxjob #2102: arXiv papers whose current version is a withdrawal, from DataCite''s Withdrawn date at the record''s current version. doi = cleaned lowercase 10.48550/arxiv.<id>; pmh = lowercase oai:arxiv.org:<id>. Rebuilt by CreateSuperLocations each run; read there (pdf_url) and by CreateWorksBase (is_pulled).'
AS
WITH latest AS (
  SELECT lower(id) AS doi, attributes.version AS version, attributes.dates AS dates,
    ROW_NUMBER() OVER (PARTITION BY lower(id) ORDER BY attributes.updated DESC, ingested_at DESC) AS rn
  FROM openalex.datacite.datacite_items
  WHERE lower(id) LIKE '10.48550/arxiv.%'
)
SELECT
  regexp_replace(doi, '[^a-zA-Z0-9./-]', '') AS doi,
  concat('oai:arxiv.org:', regexp_extract(doi, '^10\\.48550/arxiv\\.(.+)$', 1)) AS pmh,
  max(w.date) AS withdrawn_at,
  max(w.dateInformation) AS withdrawn_note
FROM latest LATERAL VIEW explode(filter(dates, d -> d.dateType = 'Withdrawn')) t AS w
WHERE rn = 1 AND split_part(w.dateInformation, ';', 1) = concat('v', version)
GROUP BY doi

In [ ]:
-- oxjob #2434: PDF rows filed under a lowercased repository record id. The Feb-Mar 2025 legacy parse stored every record
-- id in lowercase (oai:repec:..., oai:citeseerx.psu:..., oai:dialnet.unirioja.es:art0001420092), and #984's re-parse of
-- those files (27-31 Aug 2026) carried the ids into pdf_works, while the records read oai:RePEc:..., oai:CiteSeerX.psu:...,
-- ...:ART0001420092. pdf_urls_for_repo and other_fields_for_repo match a PDF row to its record by exact id, so these rows
-- never reached their record: no PDF link, and no open access from the PDF (#1150). A PDF row takes its record's spelling
-- when its id matches exactly one repository record ignoring case and that record has no PDF row under its own spelling,
-- so no record that has a PDF today switches to another. 23 Nov 2026: 2,610,532 records (2,927,558 matched ignoring case;
-- 316,930 have their own PDF rows and keep them). Landing-page rows are left as they are. Rebuilt each run; read below by
-- pdf_urls_for_repo and other_fields_for_repo. Revert: git revert the commit (the readers go with it), then DROP TABLE
-- openalex.works.pdf_pmh_case_fix; never drop it while a reader is live.
CREATE OR REPLACE TABLE openalex.works.pdf_pmh_case_fix
COMMENT 'oxjob #2434: PDF rows whose repository record id differs from the record''s only by case (the Feb-Mar 2025 legacy parse lowercased it). pdf_pmh = the id as the PDF row has it; record_pmh = the one repository record it matches ignoring case, which has no PDF row of its own spelling. Rebuilt by CreateSuperLocations each run; read there by pdf_urls_for_repo and other_fields_for_repo.'
AS
WITH pdf AS (
  SELECT DISTINCT get(filter(ids, x -> x.namespace = 'pmh').id, 0) AS pdf_pmh
  FROM openalex.works.locations_parsed
  WHERE provenance = 'pdf' AND array_contains(ids.namespace, 'pmh')
),
record AS (
  SELECT lower(native_id) AS pmh_lower, max(native_id) AS record_pmh
  FROM openalex.works.locations_parsed
  WHERE provenance IN ('repo', 'repo_backfill')
  GROUP BY 1
  HAVING count(DISTINCT native_id) = 1
)
SELECT p.pdf_pmh, r.record_pmh
FROM pdf p
JOIN record r ON r.pmh_lower = lower(p.pdf_pmh)
LEFT ANTI JOIN pdf own ON own.pdf_pmh = r.record_pmh
WHERE r.record_pmh <> p.pdf_pmh

In [ ]:
create
or replace table openalex.works.superlocations as (
  with
  -- Not-an-abstract filter (oxjob #1800). Crossref deposits, repository and DataCite records, PDF parses and
  -- landing pages parsed before 24 Oct 2026 carry the same placeholders and boilerplate Parseland no longer emits
  -- ("No abstract is available for this article.", "-", "Abstract", "N/A", site blurbs, SEO citation lines).
  -- openalex.abstracts.not_abstract_texts holds every such text, found with parseland-lib's is_not_abstract() by
  -- notebooks/maintenance/FlagLocationNotAbstracts, keyed by xxhash64 + length so a new record carrying a known
  -- junk text is caught too. NULLed here, before the per-field priorities below, so a Crossref placeholder no
  -- longer hides the landing page's real abstract: selection falls through to the next source, or to none.
  -- Kept: per-record templates (SEO citation lines, "Presenter: ...") outside landing pages and Crossref, where
  -- they also match real descriptions; and openalex.abstracts.not_abstract_owner_rows, the records a hand-read
  -- list text describes (a book's own blurb), precomputed by the same notebook.
  -- Revert: point the six readers below back at openalex.works.locations_parsed (or TRUNCATE both tables; never
  -- DROP them while this reads them).
  -- Repository PDF texts (oxjob #1800, 27 Oct 2026). A repository record with no description falls back to its
  -- PDF's GROBID 'abstract' below (repo > landing page > PDF), and GROBID often takes body text: a title page,
  -- acknowledgements, a table of contents, an OCR'd page. openalex.abstracts.not_abstract_pdf_texts lists the
  -- (pmh, text) pairs that Jev reads as not an abstract (p(abstract) < 0.05) and that carry no abstract heading
  -- anywhere in the text (bar: Opus 5.5 on a fresh random draw of the flagged rows). pdf rows only, keyed on the
  -- record's pmh id + xxhash64 + length (one row per key). Revert: TRUNCATE that table, or drop the np join.
  -- Same-work copies (oxjob #1800, 28 Oct 2026): the identical text can reach the same work through a second pdf row (the
  -- PDF fetched by the work's DOI, or via another of its repository records). Rows with doi set (pmh null) key a DOI's
  -- pdf row; another record's pmh is just another pmh row. Built only from pdf rows linked to the flagged work.
  -- Revert: DELETE those rows (label_version 'copy: ...'), or drop the npd join.
  -- Peer reviews' borrowed abstracts (oxjob #1790, 29 Oct 2026). Crossref deposits peer reviews (F1000Research / Gates /
  -- HRB / NIHR reviewer reports, Copernicus referee comments and replies, eLife reviews and assessments, PeerJ, SciPost,
  -- Cambridge Prisms ...) without an abstract, and the review's landing page or PDF carries the REVIEWED article's
  -- abstract, so the review showed it. openalex.abstracts.peer_review_borrowed_abstracts lists the landing_page and pdf
  -- rows of Crossref peer-review DOIs whose text is word for word (letters and digits, case-free) another work's abstract
  -- (not a work titled like a review, which may be the review's own copy). Reviews whose page carries their own text
  -- (Qeios, MIT Rapid Reviews, most eLife assessments) match nothing and keep it. Keyed provenance + native_id + xxhash64
  -- + length. Bar: 40 of 40 random rows matched the reviewed article (#1790 work/pr_borrowed_draw.tsv).
  -- Revert: TRUNCATE that table, or drop the pb join.
  -- Different-item pages (oxjob #1790, 1 Nov 2026). Some stored DOI landing pages show ANOTHER item: an OECD / UN /
  -- WTO chapter's DOI lands on its book's page, a JAMA / Karger / Cell Press DOI on a neighbouring article, a JoVE video or
  -- F1000 data DOI on its article. The page's abstract (and the PDF it links) is that other item's, so the work showed
  -- it (21,408 works on 1 Nov). openalex.abstracts.different_item_abstracts lists the landing_page and pdf rows whose
  -- page names a different Crossref DOI in its DOI tag, shows that DOI's title (not the work's), and whose text is word
  -- for word that DOI's own abstract in OpenAlex. Keyed provenance + native_id + xxhash64 + length, so a re-fetched
  -- page with new text is not affected. Bar: 39 of 40 random rows another item's text (#1790 EXPLORE § 20).
  -- Revert: TRUNCATE that table, or drop the di join.
  locations_parsed_checked as (
    select
      * except (abstract, abstract_inverted_index, _not_abstract),
      if(_not_abstract, null, abstract) as abstract,
      if(_not_abstract, null, abstract_inverted_index) as abstract_inverted_index
    from (
      select /*+ BROADCAST(na), BROADCAST(own), BROADCAST(np), BROADCAST(npd), BROADCAST(pb), BROADCAST(di) */
        lp.*,
        (na.abstract_hash is not null
          and (na.reason <> 'template' or lp.provenance in ('crossref', 'landing_page'))
          and own.native_id is null)
          or np.pmh is not null
          or npd.doi is not null
          or pb.native_id is not null
          or di.native_id is not null as _not_abstract
      from
        openalex.works.locations_parsed lp
        left join openalex.abstracts.not_abstract_texts na
          on na.abstract_hash = xxhash64(lp.abstract) and na.text_len = length(lp.abstract)
        left join openalex.abstracts.not_abstract_owner_rows own
          on own.abstract_hash = na.abstract_hash and own.provenance = lp.provenance and own.native_id = lp.native_id
        left join openalex.abstracts.not_abstract_pdf_texts np
          on lp.provenance = 'pdf'
          and np.pmh = get(filter(lp.ids, x -> x.namespace = 'pmh').id, 0)
          and np.abstract_hash = xxhash64(lp.abstract) and np.text_len = length(lp.abstract)
        left join (select doi, abstract_hash, text_len from openalex.abstracts.not_abstract_pdf_texts where doi is not null) npd
          on lp.provenance = 'pdf'
          and npd.doi = get(filter(lp.ids, x -> x.namespace = 'doi').id, 0)
          and npd.abstract_hash = xxhash64(lp.abstract) and npd.text_len = length(lp.abstract)
        left join openalex.abstracts.peer_review_borrowed_abstracts pb
          on pb.provenance = lp.provenance and pb.native_id = lp.native_id
          and pb.abstract_hash = xxhash64(lp.abstract) and pb.text_len = length(lp.abstract)
        left join openalex.abstracts.different_item_abstracts di
          on di.provenance = lp.provenance and di.native_id = lp.native_id
          and di.abstract_hash = xxhash64(lp.abstract) and di.text_len = length(lp.abstract)
    )
  ),

  -- crossref landing page url
  landing_page_urls_for_crossref as (
    select
      doi,
      landing_page_url
    from (
      select
        get(filter(ids, x -> x.namespace = 'doi').id, 0) as doi,
        native_id as landing_page_url,
        row_number() over(
          partition by get(filter(ids, x -> x.namespace = "doi").id, 0)
          order by 
            case 
              -- Highest priority: official DOI links
              when contains(native_id, "doi.org/") then 1
              -- Publisher domain patterns - high priority
              when contains(native_id, 'elsevier.com') then 2
              when contains(native_id, 'springer.com') then 2
              when contains(native_id, 'wiley.com') then 2
              when contains(native_id, 'sagepub.com') then 2
              when contains(native_id, 'ieee.org') then 2
              when contains(native_id, 'tandfonline.com') then 2
              when contains(native_id, 'oup.com') then 2
              when contains(native_id, 'nature.com') then 2
              when contains(native_id, 'acs.org') then 2
              -- Medium priority: institutional repositories
              when contains(native_id, 'edu/') then 3
              when contains(native_id, 'ac.uk') then 3
              when contains(native_id, 'gov/') then 3
              -- Avoid PDF links in landing pages
              when contains(lower(native_id), '.pdf') then 10
              -- Everything else
              else 5
            end asc,
            -- Use updated date as tiebreaker
            updated_date DESC NULLS LAST,
            -- Final alphabetical tiebreaker for complete determinism
            native_id ASC NULLS LAST
        ) as row_num
      from
        openalex.works.locations_parsed
      where
        provenance = 'landing_page'
        and array_contains(ids.namespace, 'doi')
    )
    where row_num = 1
  ),

  -- crossref pdf url
  pdf_urls_for_crossref as (
    select
      doi,
      pdf_url,
      pdf_s3_id,
      grobid_s3_id
    from (
      select
        get(filter(ids, x -> x.namespace = "doi").id, 0) as doi,
        -- oxjob #1935: a pdf row filed under an image link never gives the publisher copy its PDF link. On ScienceDirect
        -- pages Parseland tags the graphical abstract or a figure (ars.els-cdn.com/content/image/...-ga1_lrg.jpg) as the
        -- pdf link and TaxiCab files the fetched file under it, so the image became the work's pdf_url (49,110 works,
        -- 43,687 as best OA, 13 Nov; 19 ticket works checked 15 Nov: all 200 image/jpeg). Such a row sorts after every
        -- non-image pdf row of the DOI (first order term below); if it still wins, pdf_url is null. Its file ids and the
        -- copy's is_oa are untouched (#926 Defect D: is_oa is not decided from the URL).
        -- Revert: this CASE back to `native_id as pdf_url,` and drop the #1935 order term.
        case when (native_id rlike '(?i)\\.(jpe?g|png|gif|webp|bmp|tiff?|svg)([?#]|$)' or native_id like '%ars.els-cdn.com/content/image/%') then null else native_id end as pdf_url,
        get(filter(ids, x -> x.namespace = 'docs.pdf').id, 0) as pdf_s3_id,
        get(filter(ids, x -> x.namespace = 'docs.parsed-pdf').id, 0) as grobid_s3_id,
        row_number() over (
          partition by get(filter(ids, x -> x.namespace = "doi").id, 0)
          order by
            -- oxjob #1935: a real PDF link (any version) beats an image link
            case when (native_id rlike '(?i)\\.(jpe?g|png|gif|webp|bmp|tiff?|svg)([?#]|$)' or native_id like '%ars.els-cdn.com/content/image/%') then 1 else 0 end asc,
            -- First prioritize by version
            case
              when version = 'publishedVersion' then 1
              when version = 'acceptedVersion' then 2
              when version = 'submittedVersion' then 3
              else 4
            end asc,
            -- Then by whether it has a grobid id
            get(filter(ids, x -> x.namespace = 'docs.parsed-pdf').id, 0) nulls last,
            -- Then by URL pattern - prioritize official publisher domains
            case
              when contains(native_id, 'elsevier.com') then 1
              when contains(native_id, 'springer.com') then 1 
              when contains(native_id, 'wiley.com') then 1
              when contains(native_id, 'sagepub.com') then 1
              when contains(native_id, 'ieee.org') then 1
              when contains(native_id, 'tandfonline.com') then 1
              when contains(native_id, 'oup.com') then 1
              when contains(native_id, 'nature.com') then 1
              when contains(native_id, 'acs.org') then 1
              -- Lower priority for less reliable sources
              when contains(native_id, 'researchgate') then 3
              when contains(native_id, 'academia.edu') then 4
              else 2
            end asc,
            -- Date as tiebreaker
            updated_date DESC NULLS LAST,
            -- Final alphabetical tiebreaker for complete determinism
            native_id ASC NULLS LAST
        ) as row_num
      from
        openalex.works.locations_parsed
      where
        -- oxjob #757: a pdf row that does not attest OA (front matter) never supplies pdf_url
        provenance = 'pdf' and is_oa and array_contains(ids.namespace, 'doi')
    ) 
    where row_num = 1 
  ),
  
  other_fields_for_crossref as (
    select
      case
        when provenance = 'crossref' then native_id
        else get(filter(ids, x -> x.namespace = "doi").id, 0)
      end as best_doi,
      provenance,
      -- oxjob #2277 (ZD 24348): a page or PDF parse's junk licence (the denylist in superlocations_cleaned below; keep the
      -- two lists equal) drops out BEFORE the licence is picked, not only after. A PDF parse carries the parser's own
      -- licence (gpl-2, gpl-3, apache-2.0) and outranked the DOI's Crossref licence, then the denylist nulled it, so the
      -- copy served no licence at all (10.1021/pr4008078: PDF apache-2.0 beat Crossref's ACS other-oa; bronze, not
      -- hybrid). 24 Nov 2026: 1,094 Crossref copies regain their Crossref licence (cc-by 904). Revert: git revert the
      -- commit (it also adds one branch to other_fields_for_repo's licence CASE; the IRDB branch there stays).
      case
        when provenance in ('landing_page', 'pdf')
             and license in ('False', 'unspecified-oa', 'gpl', 'gpl-2', 'gpl-3', 'apache-2.0') then null
        else license
      end as license,
      is_oa,
      case
        when provenance = 'landing_page' then 1
        when provenance = 'pdf' then 2
        when provenance = 'crossref' then 3
      end as license_priority,
      references,
      case
        when provenance = 'crossref' then 1
        -- oxjob #2215 (16 Nov 2026): a landing page's reference list (LandingPage.py keeps only lists Parseland proves
        -- whole) fills a work only when Crossref and the PDF give none. A PDF list that GROBID left empty (22.4M pdf rows)
        -- counts as none: it ranks below the page's list and below another PDF's non-empty list (2.6K DOI works newly
        -- take an older PDF's list over the newest PDF's empty one). With no other list it still wins, as before.
        -- Revert: drop the landing_page and empty-pdf lines here and in repo_superlocations' copy below.
        when provenance = 'pdf' and size(references) = 0 then 4
        when provenance = 'pdf' then 2
        when provenance = 'landing_page' then 3
      end as references_priority,
      authors,
      case
        when provenance = 'crossref' then 1
        when provenance = 'landing_page' then 2
        when provenance = 'pdf' then 3
      end as authors_priority,
      -- Junk-abstract denylist (oxjob 682): known parser boilerplate must never win
      -- abstract selection. NULLed here so these rows drop out of the abstract_list /
      -- abstract_inverted_index_list filters below and selection falls through to the
      -- next-priority source, or yields no abstract at all.
      CASE WHEN abstract LIKE 'Click to increase image size%'
             OR abstract LIKE 'An abstract is not available for this content so a preview%'
             OR abstract LIKE '%A summary is not available for this content so a preview%'
             OR abstract LIKE 'Advertisements that appeared within the print issues of Chem. Eng. News%'
           THEN NULL ELSE abstract END AS abstract,
      CASE WHEN abstract LIKE 'Click to increase image size%'
             OR abstract LIKE 'An abstract is not available for this content so a preview%'
             OR abstract LIKE '%A summary is not available for this content so a preview%'
             OR abstract LIKE 'Advertisements that appeared within the print issues of Chem. Eng. News%'
           THEN NULL ELSE abstract_inverted_index END AS abstract_inverted_index,
      case
        when provenance = 'crossref' then 1
        when provenance = 'landing_page' then 2
        when provenance = 'pdf' then 3
      end as abstract_priority,
      updated_date,
      native_id  -- Added for deterministic sorting
    from
      locations_parsed_checked
    where
      provenance in (
        'crossref',
        'landing_page',
        'pdf'
      )
  ),
  
  staging_other_fields_for_crossref as (
    select
      best_doi,
      exists(array_sort(collect_list(struct(is_oa, updated_date, native_id)), 
           (a, b) -> case 
                       when a.updated_date > b.updated_date then -1 
                       when a.updated_date < b.updated_date then 1
                       when a.native_id < b.native_id then -1
                       else 1 
                     end).is_oa, 
           x -> x = True) as is_oa,
    -- Sort each list deterministically before filtering
    filter(
      array_sort(
        collect_list(struct(license, license_priority, updated_date, native_id)),
        (a, b) -> case 
                    when a.license_priority < b.license_priority then -1
                    when a.license_priority > b.license_priority then 1
                    when a.updated_date > b.updated_date then -1 
                    when a.updated_date < b.updated_date then 1
                    when a.native_id < b.native_id then -1
                    else 1 
                  end
      ),
      x -> nullif(x.license, "") is not null
    ) as license_list,
    
    -- Sort authors list deterministically
    array_sort(
      collect_list(struct(authors, authors_priority, updated_date, native_id)),
      (a, b) -> case 
                  when a.authors_priority < b.authors_priority then -1
                  when a.authors_priority > b.authors_priority then 1
                  when a.updated_date > b.updated_date then -1 
                  when a.updated_date < b.updated_date then 1
                  when a.native_id < b.native_id then -1
                  else 1 
                end
    ) as authors_list,
    
    -- Sort references list deterministically
    filter(
      array_sort(
        collect_list(struct(references, references_priority, updated_date, native_id)),
        (a, b) -> case 
                    when a.references_priority < b.references_priority then -1
                    when a.references_priority > b.references_priority then 1
                    when a.updated_date > b.updated_date then -1 
                    when a.updated_date < b.updated_date then 1
                    when a.native_id < b.native_id then -1
                    else 1 
                  end
      ),
      x -> x.references is not null
    ) as references_list,
    filter(
      collect_list(struct(abstract, abstract_priority, updated_date, native_id)),
      x -> x.abstract is not null and trim(x.abstract) != ''
    ) as abstract_list,
    filter(
      collect_list(struct(abstract_inverted_index, abstract_priority, updated_date, native_id)),
      x -> x.abstract_inverted_index is not null
    ) as abstract_inverted_index_list
    from
      other_fields_for_crossref
    where
      best_doi is not null
    group by
      best_doi
  ),
  
  best_other_fields_for_crossref as (
    -- get the highest priority value per license and author
    select
      best_doi,
      is_oa,
      -- Get first license after deterministic sort
      get(
        array_sort(
          filter(
            license_list,
            x -> x.license_priority = array_min(license_list.license_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.license < b.license then -1
                      else 1 
                    end
        ).license,
        0
      ) as license,
      
      -- Get first authors after deterministic sort
      get(
        array_sort(
          filter(
            authors_list,
            x -> x.authors_priority = array_min(authors_list.authors_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.native_id < b.native_id then -1
                      else 1 
                    end
        ).authors,
        0
      ) as authors,
      
      -- Get first references after deterministic sort
      get(
        array_sort(
          filter(
            references_list,
            x -> x.references_priority = array_min(references_list.references_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.native_id < b.native_id then -1
                      else 1 
                    end
        ).references,
        0
      ) as references,
      get(
        array_sort(
          filter(abstract_list, x -> x.abstract_priority = array_min(abstract_list.abstract_priority)),
          (a, b) -> case
            when a.updated_date > b.updated_date then -1
            when a.updated_date < b.updated_date then 1
            when a.native_id < b.native_id then -1
            else 1
          end
        ).abstract, 0
      ) as abstract,
      get(
        array_sort(
          filter(abstract_inverted_index_list, x -> x.abstract_priority = array_min(abstract_inverted_index_list.abstract_priority)),
          (a, b) -> case
            when a.updated_date > b.updated_date then -1
            when a.updated_date < b.updated_date then 1
            when a.native_id < b.native_id then -1
            else 1
          end
        ).abstract_inverted_index, 0
      ) as abstract_inverted_index
    from
      staging_other_fields_for_crossref
  ),
  
  -- oxjob #2449 (19 Nov 2026): PMC holds the journal's published version of many Crossref DOIs under a Creative Commons
  -- licence taken from the publisher's own article XML (openalex.pubmed_central.pmc_oa_articles, newest version of each
  -- article), while the DOI's landing page, PDF and Crossref deposit give none. So a journal copy the publisher serves free
  -- read free with no licence (the work bronze, not hybrid) and the license filter missed the work. PMC's licence is the
  -- article's own (policy ruling 22 Oct 2026, 'Trust the licence'), so it fills the licence of a journal copy that is
  -- already open when no other source gives one. It never opens a closed copy: on a fresh draw, 7 of 30 such DOIs led to
  -- a dead or moved site or a paywall. Left out: author manuscripts (PMC's flag: their licence is the manuscript's, #1860),
  -- historical OCR scans (a scanned back issue's PMC licence is not the publisher's), DOIs whose PMC records disagree on
  -- the licence or whose newest PMC read failed, and the hold-outs at the join below.
  -- Bar (publisher pages, fresh draw of 30 from what this changes): open journal copies free on 27 of 30; where a page
  -- names a licence, PMC's matched it on 33 of 36 in the first draw.
  -- Revert: drop this CTE and pmc_doaj_licensed_dois, the pl / dh joins and the pl term in crossref_superlocations.
  pmc_publisher_licences as (
    select doi, max(license) as license
    from (
      select
        lower(trim(newest.doi)) as doi,
        lower(replace(nullif(trim(newest.license_code), ''), ' ', '-')) as license,
        newest.is_manuscript,
        newest.is_historical_ocr
      from (
        -- one row per article: its newest version, newest read (one max_by, so every field comes from the same row); a newest
        -- read that failed (the bucket no longer serves the article, ~1.2K) gives no licence rather than an older read's
        select max_by(struct(doi, license_code, is_manuscript, is_historical_ocr, error), struct(version, fetched_at, key)) as newest
        from openalex.pubmed_central.pmc_oa_articles
        where pmcid rlike '^PMC[0-9]+$'
        group by pmcid
      )
      where newest.error is null
    )
    where is_manuscript = false
      and is_historical_ocr = false
      and nullif(doi, '') is not null
    group by doi
    -- every published-version record PMC holds for the DOI states the same licence (none counts as a licence), and it is CC
    having count(distinct coalesce(license, '(none)')) = 1
      and max(license) in ('cc-by', 'cc-by-nc', 'cc-by-nc-nd', 'cc-by-nc-sa', 'cc-by-nd', 'cc-by-sa', 'cc0')
  ),

  -- oxjob #2449: a journal whose DOAJ record gives a default licence keeps it (CreateWorksBase fills it): PMC's licence
  -- matched the publisher page on 6 of 7 such articles, under the bar. Keyed on the Crossref record's ISSNs.
  pmc_doaj_licensed_dois as (
    select distinct cl.native_id
    from (
      select c.native_id, issn
      from locations_parsed_checked c
        join pmc_publisher_licences p on c.native_id = p.doi
        lateral view explode(filter(c.ids, x -> x.namespace in ('issn', 'pissn', 'eissn')).id) t as issn
      where c.provenance = 'crossref'
    ) cl
    join (
      select distinct issn
      from openalex.sources.sources lateral view explode(issns) t as issn
      where is_in_doaj and nullif(doaj_license, '') is not null
    ) d on d.issn = cl.issn
  ),

  -- oxjob #2504 (22 Nov 2026): a Crossref copy's doi.org PDF link that only leads to the publisher's page becomes null.
  -- 3.26M Crossref copies took a pdf row filed under the DOI itself (legacy harvests), so pdf_url was the doi.org link.
  -- Unlike DataCite's (#1646, 4 of 100 PDFs), most Crossref ones DO serve a PDF: the publisher registered the file as
  -- the DOI's target (59 of 100 random ones fetched were PDFs). So the link goes only when nothing says it is a file:
  -- the DOI's registered URL (Crossref resource.primary.URL) has no file word in it, TaxiCab never fetched the doi.org
  -- link as a PDF, and the prefix is not one of four publishers whose registered page URL serves or redirects to the
  -- PDF in every sampled case (OMICS 10.4172, Jaypee 10.5005, IJRASET 10.22214, Editora Cubo 10.4322). Component DOIs
  -- (supplementary files, not in crossref_deduplicated) keep theirs. ~655K copies: of a fresh 100 it nulls, ~87 were a
  -- page, a bot wall or dead, ~10 a PDF. Stored PDF and GROBID file stay (has_content.pdf); #2182 keeps oa_status.
  -- Revert: drop this CTE, the pp join and the CASE in crossref_superlocations (back to `p.pdf_url,`).
  crossref_doi_pdf_url_is_a_page as (
    select p.doi
    from pdf_urls_for_crossref p
    join openalex.crossref.crossref_deduplicated r on r.native_id = p.doi
    left anti join (
      select distinct lower(regexp_replace(url, '(?i)^https?://(dx\\.)?doi\\.org/', '')) as doi
      from openalex.taxicab.taxicab_results
      where url rlike '^(?i)https?://(dx\\.)?doi\\.org/' and content_type like '%pdf%'
    ) t on t.doi = lower(p.doi)
    where p.pdf_url rlike '^(?i)https?://(dx\\.)?doi\\.org/'
      and r.resource.primary.URL is not null
      and not r.resource.primary.URL rlike '(?i)pdf|download|fileserve|getfile|bitstream|/purl/|type=full|fulltext|full_text|attachment'
      and split_part(p.doi, '/', 1) not in ('10.4172', '10.5005', '10.22214', '10.4322')
  ),

  crossref_superlocations as (
    select
      c.provenance,
      c.native_id,
      c.native_id_namespace,
      c.title,
      c.normalized_title,
      coalesce(a.authorships, c.authors) as authors,
      c.ids,
      c.raw_type,
      c.type,
      c.version,
      -- oxjob #2449: PMC's publisher licence fills an open journal copy no other source licenses (pmc_publisher_licences)
      coalesce(nullif(o.license, ""), pl.license) as license,
      c.language,
      c.published_date,
      c.created_date,
      c.updated_date,
      c.issue,
      c.volume,
      c.first_page,
      c.last_page,
      c.is_retracted,
      COALESCE(o.abstract, c.abstract) as abstract,
      c.source_name,
      c.publisher,
      c.funders,
      o.references,
      c.urls,
      c.mesh,
      o.is_oa,
      cast(null as string) as endpoint_id,
      COALESCE(o.abstract_inverted_index, c.abstract_inverted_index) as abstract_inverted_index,
      c.authors_exist,
      c.affiliations_exist,
      c.is_corresponding_exists,
      c.best_doi,
      c.merge_key,
      case when pp.doi is not null then null else p.pdf_url end as pdf_url, -- oxjob #2504
      l.landing_page_url,
      p.pdf_s3_id,
      p.grobid_s3_id,
      c.ingested_at
    from
      locations_parsed_checked c
      left join pdf_urls_for_crossref p on c.native_id = p.doi
      left join crossref_doi_pdf_url_is_a_page pp on c.native_id = pp.doi -- oxjob #2504
      left join landing_page_urls_for_crossref l on c.native_id = l.doi
      left join best_other_fields_for_crossref o on c.native_id = o.best_doi
      left join pmc_doaj_licensed_dois dh on c.native_id = dh.native_id -- oxjob #2449
      left join pmc_publisher_licences pl on c.native_id = pl.doi -- oxjob #2449
        and coalesce(o.is_oa, false) -- only a copy already open; never opens one
        and dh.native_id is null
        -- Held out by the publisher-page bar (19 Nov 2026): Rockefeller articles before 2010 are paywalled on the
        -- publisher's site though PMC holds them CC BY-NC-SA (21 of 21 pages); Bentham, Dustri, JADPRO, CORR and Clinical
        -- and Experimental Dermatology pages were paywalled; CDC journals are public domain (PMC's code reads CC BY).
        and not (split_part(c.native_id, '/', 1) in ('10.1083', '10.1084', '10.1085') and coalesce(year(c.published_date), 0) < 2010)
        and split_part(c.native_id, '/', 1) not in ('10.2174', '10.5414', '10.6004', '10.15585', '10.5888', '10.3201')
        and c.native_id not like '10.1007/s11999-%'
        and c.native_id not like '10.1111/ced.%'
      left join openalex.works.crossref_super_authorships a 
      on c.native_id = a.doi
    where
      provenance = 'crossref'
  ),
  
  landing_page_urls_for_repo as (
    select
      pmh,
      landing_page_url
    from (
      select
        get(filter(ids, x -> x.namespace = 'pmh').id, 0) as pmh,
        native_id as landing_page_url,
        row_number() over (
          partition by get(filter(ids, x -> x.namespace = 'pmh').id, 0)
          order by 
            -- Prioritize specific repository URL patterns
            case
              when contains(native_id, 'arxiv.org') then 1
              when contains(native_id, 'biorxiv.org') then 1
              when contains(native_id, 'medrxiv.org') then 1
              when contains(native_id, 'ssrn.com') then 2
              when contains(native_id, 'zenodo.org') then 2
              when contains(native_id, 'figshare.com') then 2
              when contains(native_id, 'edu/') then 3
              when contains(native_id, 'ac.uk') then 3
              when contains(native_id, 'gov/') then 3
              when contains(lower(native_id), '.pdf') then 10
              else 5
            end asc,
            updated_date DESC NULLS LAST,
            native_id ASC NULLS LAST
        ) as row_num
      from
        openalex.works.locations_parsed
      where
        provenance = 'landing_page'
        and array_contains(ids.namespace, 'pmh')
    )
    where row_num = 1
  ),
  
  -- oxjob #933: a PDF fetched from an OSTI purl is keyed by the biblio URL or the DOI
  -- when taxicab queued it from the publisher side, never by the repo record's pmh id.
  -- The purl's numeric id IS the OSTI record id, so key those rows back onto it.
  pdf_urls_for_repo as (
    select
      pmh,
      pdf_url,
      pdf_s3_id,
      grobid_s3_id
    from (
      select /*+ BROADCAST(cf) */  -- oxjob #2434
        coalesce(
          cf.record_pmh,  -- oxjob #2434: a lowercased record id takes its record's spelling (cell above)
          get(filter(ids, x -> x.namespace = "pmh").id, 0),
          case when native_id rlike 'osti\\.gov/servlets/purl/[0-9]+'
               then concat('oai:osti.gov:', regexp_extract(native_id, 'servlets/purl/([0-9]+)', 1)) end
        ) as pmh,
        -- oxjob #1935: an image is never a repository copy's PDF link either (same rule as pdf_urls_for_crossref above:
        -- sorts after every non-image pdf row of the record; if it still wins, pdf_url is null; file ids and is_oa kept).
        -- Revert: this CASE back to `native_id as pdf_url,` and drop the #1935 order term below.
        -- oxjob #2380: NVA's file link answers JSON naming a presigned S3 URL (Taxicab follows it), so it is never shown
        -- as a pdf_url either; the stored file ids stay. Revert: drop the api.nva.unit.no term.
        case when (native_id rlike '(?i)\\.(jpe?g|png|gif|webp|bmp|tiff?|svg)([?#]|$)' or native_id like '%ars.els-cdn.com/content/image/%'
                   or native_id rlike '^https://api\\.nva\\.unit\\.no/publication/[^/]+/filelink/') then null else native_id end as pdf_url,
        get(filter(ids, x -> x.namespace = 'docs.pdf').id, 0) as pdf_s3_id,
        get(filter(ids, x -> x.namespace = 'docs.parsed-pdf').id,0) as grobid_s3_id,
        row_number() over (
          partition by coalesce(
            cf.record_pmh,
            get(filter(ids, x -> x.namespace = "pmh").id, 0),
            case when native_id rlike 'osti\\.gov/servlets/purl/[0-9]+'
                 then concat('oai:osti.gov:', regexp_extract(native_id, 'servlets/purl/([0-9]+)', 1)) end
          )
          order by
            -- oxjob #1935: a real PDF link (any version) beats an image link
            case when (native_id rlike '(?i)\\.(jpe?g|png|gif|webp|bmp|tiff?|svg)([?#]|$)' or native_id like '%ars.els-cdn.com/content/image/%') then 1 else 0 end asc,
            -- First by version
            case
              when version = 'publishedVersion' then 1
              when version = 'acceptedVersion' then 2
              when version = 'submittedVersion' then 3
              else 4
            end asc,
            -- Then by whether it has a grobid id 
            get(filter(ids, x -> x.namespace = 'docs.parsed-pdf').id, 0) nulls last,
            -- Then by URL pattern
            case
              when contains(native_id, 'arxiv.org') then 1
              when contains(native_id, 'biorxiv.org') then 1
              when contains(native_id, 'medrxiv.org') then 1
              when contains(native_id, 'edu/') then 2
              when contains(native_id, 'ac.uk') then 2
              when contains(native_id, 'gov/') then 2
              else 3
            end asc,
            -- Date as tiebreaker
            updated_date DESC NULLS LAST,
            -- Final alphabetical tiebreaker for complete determinism
            native_id ASC NULLS LAST
        ) as row_num
      from
        openalex.works.locations_parsed
        left join openalex.works.pdf_pmh_case_fix cf  -- oxjob #2434
          on cf.pdf_pmh = get(filter(ids, x -> x.namespace = "pmh").id, 0)
      where
        provenance = 'pdf' and is_oa
        and (array_contains(ids.namespace, 'pmh') or native_id rlike 'osti\\.gov/servlets/purl/[0-9]+')
        -- oxjob #1854: a Digital Commons record's "additional files" (/type/additional/viewcontent/..., old form
        -- viewcontent.cgi?...&type=additional) are never the item's full text: the platform serves the article as the
        -- record's main file. TaxiCab fetches them all and the order above can't tell them apart (the grobid key compares
        -- id values), so on 16 Nov 2026 892 of 16,214 PEARL PDF links were an additional file, 719 of them the
        -- university's deposit agreement or licence form (ZD 8738, W4404547497); ~3K such links corpus-wide. They no
        -- longer supply pdf_url; the main file does when it was fetched, else the copy has no PDF link. is_oa is
        -- untouched (other_fields_for_repo). Revert: delete these two lines.
        and not (native_id rlike '/type/additional/viewcontent/'
                 or (native_id like '%viewcontent.cgi?%' and native_id rlike '[?&]type=additional'))
    )
    where row_num = 1
  ),
  
  -- oxjob #1132 (16 Nov 2026): the version a repository record states in its own dc:type: the eu-repo token
  -- (info:eu-repo/semantics/publishedVersion), the COAR version URI, the bare token Worktribe and others emit
  -- (publishedVersion), or the short codes VoR / AM. Declarations beat inference (#805's version policy; Jason's #933
  -- ruling 3, "we trust declarations"): the record's declaration outranks the landing-page and PDF parses of the same
  -- item, which default to submittedVersion and so outvoted it (16 Nov: 11.77M repository copies served a version
  -- their own record contradicts, 10.06M of them declaring publishedVersion). Several different declarations: the
  -- lowest (Durham lists accepted + published on records whose file is the accepted manuscript). Read from
  -- openalex.repo.repo_parsed (every harvest; repo_works keeps one dc:type only), the newest harvest that declares one.
  -- PMC / Europe PMC keep their own rule (#1828, #2112). Records only in the pre-2025 backfill are not covered.
  -- Revert: drop this CTE, its join and the two #1132 lines in other_fields_for_repo.
  repo_declared_version as (
    select native_id as rdv_native_id, max_by(declared_version, ingested_at) as declared_version
    from (
      select
        native_id,
        ingested_at,
        case
          when exists(raw_native_types, x -> lower(trim(x)) rlike '^(info:eu-repo/semantics/)?(submittedversion|draft)$|^submitted[ _]version$|^(smur|ao)$|purl\\.org/coar/version/(c_b1a7d7d4d402bcce|c_71e4c1898caa6e32)$')
            then 'submittedVersion'
          when exists(raw_native_types, x -> lower(trim(x)) rlike '^(info:eu-repo/semantics/)?acceptedversion$|^(am|aam)$|^(author )?accepted (manuscript|version)$|^am - accepted manuscript$|purl\\.org/coar/version/(c_ab4af688f83e57aa|c_fa2ee174bc00049f)$')
            then 'acceptedVersion'
          when exists(raw_native_types, x -> lower(trim(x)) rlike '^(info:eu-repo/semantics/)?(published|updated)version$|^(vor|cvor|evor)$|^(published|publisher(&apos;|\')?s?) ?version$|^published_version$|^vor - version of record$|purl\\.org/coar/version/(c_970fb48d4fbd8a85|c_e19f295774971610|c_dc82b40f9837b551)$')
            -- oxjob #1132 (17 Nov 2026): a record that also types itself a preprint (eu-repo or COAR resource type)
            -- is a preprint as its server or repository published it (engrXiv, DESY, RWTH, Juelich, TU/e, ORA; 17.6K
            -- records went submitted -> published on night 2026-11-17): a preprint is the submitted version. Typed
            -- preprint + accepted stays accepted (some repositories type postprints 'preprint'). Revert: 'publishedVersion'.
            then case
              when exists(raw_native_types, x -> lower(trim(x)) rlike '^(info:eu-repo/semantics/|doc-type:)?preprint$|purl\\.org/coar/resource_type/c_816b$')
                then 'submittedVersion'
              else 'publishedVersion'
            end
        end as declared_version
      from (
        select native_id, ingested_at, raw_native_types from openalex.repo.repo_parsed
        -- oxjob #1960 (19 Nov 2026): records re-read by Repo Reparse (openalex.repo.repo_reparse_items) never pass
        -- repo_parsed, and repo_enriched_reparse keeps no dc:type, so their declaration was lost and the PDF parse's
        -- submittedVersion default won once their file was parsed (ZD 23931: ORA's accepted manuscript on W7203481173).
        -- Their dc:type is read here, keyed by the OAI header id (Repo.py re-keys only placeholder hosts); the newest
        -- copy that declares a version still wins. Dry run on night 2026-11-19's build: 19,139 copies change version,
        -- all with no declaration in repo_parsed (IRUA 14,765, ORA 4,374). Revert: drop the union all branch.
        union all
        select `ns0:header`.`ns0:identifier` as native_id, ingested_at, `ns0:metadata`.`ns1:dc`.`dc:type` as raw_native_types
        from openalex.repo.repo_reparse_items
      )
      where native_id not like 'oai:pubmedcentral.nih.gov:%'
        and native_id not like 'oai:europepmc.org:%'
        and exists(raw_native_types, x -> lower(x) rlike 'version|purl\\.org/coar/version/|^ *(draft|smur|ao|am|aam|vor|cvor|evor) *$|accepted (manuscript|version)')
    )
    where declared_version is not null
    group by native_id
  ),

  other_fields_for_repo as (
    select /*+ BROADCAST(cf) */  -- oxjob #2434
      case
        when provenance in ('repo', 'repo_backfill') then native_id
        when provenance = 'pdf' and cf.record_pmh is not null then cf.record_pmh  -- oxjob #2434 (cell above)
        when provenance = 'pdf' and native_id rlike 'osti\\.gov/servlets/purl/[0-9]+'
          then coalesce(get(filter(ids, x -> x.namespace = "pmh").id, 0),
                        concat('oai:osti.gov:', regexp_extract(native_id, 'servlets/purl/([0-9]+)', 1)))
        else get(filter(ids, x -> x.namespace = "pmh").id, 0)
      end as best_pmh,
      provenance,
      is_oa,
      -- oxjob #1132: a repository record's declared version replaces its own regex guess
      coalesce(rdv.declared_version, version) as version,
      case
        -- PMC and Europe PMC (oxjob #1828, #1344): the repository record's version comes from PMC's own set name
        -- (author manuscript or publisher deposit) and outranks the landing-page and PDF parses of the same item,
        -- which default to submittedVersion. Where they disagree, PMC's S3 metadata (is_manuscript) sides with the
        -- repository record on ~2.45M records and with the parses on 385 (27 Oct 2026).
        when provenance in ('repo', 'repo_backfill')
             and (native_id like 'oai:pubmedcentral.nih.gov:%' or native_id like 'oai:europepmc.org:%')
             and nullif(version, '') is not null then 0
        -- oxjob #1132: a version the repository record declares outranks the landing-page and PDF defaults
        when rdv.declared_version is not null then 0
        -- oxjob #891 (ZD 23514, 23576): IRDB items. IRDB.py maps the record's JPCOAR oaire:version (COAR AM / VoR) since
        -- #2147, but IRDB records are not in openalex.repo.repo_parsed, so #1132's declared version above misses them, and
        -- the landing-page and PDF parses of the same item (default submittedVersion) outvoted the record. The record's
        -- AM / VoR now ranks first. Dry run 17 Nov 2026: 1,474,704 IRDB records change version (1,371,947 submitted ->
        -- published, 92,784 submitted -> accepted); 50/50 random changes match the IRDB page's publication type.
        -- Revert: delete this branch.
        when provenance in ('repo', 'repo_backfill') and native_id like 'oai:irdb.nii.ac.jp:%'
             and version in ('acceptedVersion', 'publishedVersion') then 0
        when version = 'submittedVersion' then 1
        when version = 'acceptedVersion' then 2
        when version = 'publishedVersion' then 3
        else 4
      end as version_priority,
      -- oxjob #891 (ZD 23516, 23576): every WEKO3 / JAIRO Cloud record page embeds its search form's licence options
      -- (CC0 ... CC BY-NC-ND 4.0) in an ng-init attribute, and the landing-page parse reads that list as the page's
      -- licence (it normalises to cc-by-nc-nd), so IRDB items served a cc-by-nc-nd their record never states. A page's
      -- cc-by-nc-nd on an IRDB item no longer counts: the record's own licence (IRDB.py reads dc:rights), a PDF's, or
      -- none takes its place. Other page licences stay (13 of 30 sampled are real). is_oa is untouched (#1165).
      -- Dry run 17 Nov 2026: 1,119,002 IRDB records drop cc-by-nc-nd; 49/49 readable sampled pages show no such licence.
      -- license_priority below still reads the raw column. Revert: put back `license,`.
      case
        -- oxjob #2277: a page or PDF parse's junk licence drops out before the pick (see other_fields_for_crossref), so
        -- the record's own licence fills; a repository record's own gpl or apache-2.0 still wins and is nulled, as before.
        -- 24 Nov 2026: ~5,450 repository copies regain their record's other-oa.
        when provenance in ('landing_page', 'pdf')
             and license in ('False', 'unspecified-oa', 'gpl', 'gpl-2', 'gpl-3', 'apache-2.0') then null
        when provenance = 'landing_page' and license = 'cc-by-nc-nd'
             and get(filter(ids, x -> x.namespace = "pmh").id, 0) like 'oai:irdb.nii.ac.jp:%'
          then null
        else license
      end as license,
      case
        -- oxjob #1096 (ZD 7949): a specific licence the repository's own record states (its dc:rights) is the item's
        -- licence and outranks a parse of its page or PDF; the parses still fill a gap or a generic 'other-oa'. DSpace 7
        -- pages carry a site default or a menu of every licence beside the item's own (UNSWorks: dcterms.license
        -- by-nc-nd/3.0/au on CC BY items; TIB, Hannover: a licence picker), and the page parse read those. 15 Nov 2026:
        -- 677,752 repository copies served a parsed licence over a different specific feed licence (13,589 UNSWorks);
        -- where the page shows the item's own licence the feed matched it in 44 of 48 random copies (seeds 1096, 2096).
        -- Revert: landing_page 1, pdf 2, repo 3 (drop the first branch).
        when provenance in ('repo', 'repo_backfill') and nullif(license, '') is not null and license <> 'other-oa' then 0
        when provenance = 'landing_page' then 1
        when provenance = 'pdf' then 2
        when provenance in ('repo', 'repo_backfill') then 3
      end as license_priority,
      references,
      case
        when provenance in ('repo', 'repo_backfill') then 1
        -- oxjob #2215: as for DOIs above (32K repository records newly take an older PDF's list over an empty one)
        when provenance = 'pdf' and size(references) = 0 then 4
        when provenance = 'pdf' then 2
        when provenance = 'landing_page' then 3
      end as references_priority,
      authors,
      case
        when provenance in ('repo', 'repo_backfill') then 1
        when provenance = 'landing_page' then 2
        when provenance = 'pdf' then 3
      end as authors_priority,
      -- Junk-abstract denylist (oxjob 682): known parser boilerplate must never win
      -- abstract selection. NULLed here so these rows drop out of the abstract_list /
      -- abstract_inverted_index_list filters below and selection falls through to the
      -- next-priority source, or yields no abstract at all.
      CASE WHEN abstract LIKE 'Click to increase image size%'
             OR abstract LIKE 'An abstract is not available for this content so a preview%'
             OR abstract LIKE '%A summary is not available for this content so a preview%'
             OR abstract LIKE 'Advertisements that appeared within the print issues of Chem. Eng. News%'
           THEN NULL ELSE abstract END AS abstract,
      CASE WHEN abstract LIKE 'Click to increase image size%'
             OR abstract LIKE 'An abstract is not available for this content so a preview%'
             OR abstract LIKE '%A summary is not available for this content so a preview%'
             OR abstract LIKE 'Advertisements that appeared within the print issues of Chem. Eng. News%'
           THEN NULL ELSE abstract_inverted_index END AS abstract_inverted_index,
      case
        when provenance in ('repo', 'repo_backfill') then 1
        when provenance = 'landing_page' then 2
        when provenance = 'pdf' then 3
      end as abstract_priority,
      updated_date,
      native_id  -- Added for deterministic sorting
    from
      locations_parsed_checked
      -- oxjob #1132: the record's own declared version (repository records only)
      left join repo_declared_version rdv
        on provenance in ('repo', 'repo_backfill') and rdv.rdv_native_id = native_id
      -- oxjob #2434: a PDF row filed under a lowercased record id joins its record (cell above)
      left join openalex.works.pdf_pmh_case_fix cf
        on provenance = 'pdf' and cf.pdf_pmh = get(filter(ids, x -> x.namespace = "pmh").id, 0)
    where
      provenance in (
        'repo',
        'repo_backfill',
        'landing_page',
        'pdf'
      )
  ),
  
  staging_other_fields_for_repo as (
    select
      best_pmh,
      exists(array_sort(collect_list(struct(is_oa, updated_date, native_id)), 
           (a, b) -> case 
                       when a.updated_date > b.updated_date then -1 
                       when a.updated_date < b.updated_date then 1
                       when a.native_id < b.native_id then -1
                       else 1 
                     end).is_oa, 
           x -> x = True) as is_oa,
           
    -- Sort each list deterministically before filtering
    filter(
      array_sort(
        collect_list(struct(license, license_priority, updated_date, native_id)),
        (a, b) -> case 
                    when a.license_priority < b.license_priority then -1
                    when a.license_priority > b.license_priority then 1
                    when a.updated_date > b.updated_date then -1 
                    when a.updated_date < b.updated_date then 1
                    when a.native_id < b.native_id then -1
                    else 1 
                  end
      ),
      x -> nullif(x.license, "") is not null
    ) as license_list,
    
    -- Sort authors list deterministically
    array_sort(
      collect_list(struct(authors, authors_priority, updated_date, native_id)),
      (a, b) -> case 
                  when a.authors_priority < b.authors_priority then -1
                  when a.authors_priority > b.authors_priority then 1
                  when a.updated_date > b.updated_date then -1 
                  when a.updated_date < b.updated_date then 1
                  when a.native_id < b.native_id then -1
                  else 1 
                end
    ) as authors_list,
    
    -- Sort version list deterministically
    filter(
      array_sort(
        collect_list(struct(version, version_priority, updated_date, native_id)),
        (a, b) -> case 
                    when a.version_priority < b.version_priority then -1
                    when a.version_priority > b.version_priority then 1
                    when a.updated_date > b.updated_date then -1 
                    when a.updated_date < b.updated_date then 1
                    when a.native_id < b.native_id then -1
                    else 1 
                  end
      ),
      x -> nullif(x.version, "") is not null
    ) as version_list,
    
    -- Sort references list deterministically
    filter(
      array_sort(
        collect_list(struct(references, references_priority, updated_date, native_id)),
        (a, b) -> case 
                    when a.references_priority < b.references_priority then -1
                    when a.references_priority > b.references_priority then 1
                    when a.updated_date > b.updated_date then -1 
                    when a.updated_date < b.updated_date then 1
                    when a.native_id < b.native_id then -1
                    else 1 
                  end
      ),
      x -> x.references is not null
    ) as references_list,
    filter(
      collect_list(struct(abstract, abstract_priority, updated_date, native_id)),
      x -> x.abstract is not null and trim(x.abstract) != ''
    ) as abstract_list,
    filter(
      collect_list(struct(abstract_inverted_index, abstract_priority, updated_date, native_id)),
      x -> x.abstract_inverted_index is not null
    ) as abstract_inverted_index_list
    from
      other_fields_for_repo
    where
      best_pmh is not null
    group by
      best_pmh
  ),
  
  best_other_fields_for_repo as (
    select
      best_pmh,
      is_oa,
      
      -- Get first license after deterministic sort
      get(
        array_sort(
          filter(
            license_list,
            x -> x.license_priority = array_min(license_list.license_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.license < b.license then -1
                      else 1 
                    end
        ).license,
        0
      ) as license,
      
      -- Get first version after deterministic sort
      get(
        array_sort(
          filter(
            version_list,
            x -> x.version_priority = array_min(version_list.version_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.version < b.version then -1
                      else 1 
                    end
        ).version,
        0
      ) as version,
      
      -- Get first authors after deterministic sort
      get(
        array_sort(
          filter(
            authors_list,
            x -> x.authors_priority = array_min(authors_list.authors_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.native_id < b.native_id then -1
                      else 1 
                    end
        ).authors,
        0
      ) as authors,
      
      -- Get first references after deterministic sort
      get(
        array_sort(
          filter(
            references_list,
            x -> x.references_priority = array_min(references_list.references_priority)
          ),
          (a, b) -> case 
                      when a.updated_date > b.updated_date then -1 
                      when a.updated_date < b.updated_date then 1
                      when a.native_id < b.native_id then -1
                      else 1 
                    end
        ).references,
        0
      ) as references,
      get(
        array_sort(
          filter(abstract_list, x -> x.abstract_priority = array_min(abstract_list.abstract_priority)),
          (a, b) -> case
            when a.updated_date > b.updated_date then -1
            when a.updated_date < b.updated_date then 1
            when a.native_id < b.native_id then -1
            else 1
          end
        ).abstract, 0
      ) as abstract,
      get(
        array_sort(
          filter(abstract_inverted_index_list, x -> x.abstract_priority = array_min(abstract_inverted_index_list.abstract_priority)),
          (a, b) -> case
            when a.updated_date > b.updated_date then -1
            when a.updated_date < b.updated_date then 1
            when a.native_id < b.native_id then -1
            else 1
          end
        ).abstract_inverted_index, 0
      ) as abstract_inverted_index
    from
      staging_other_fields_for_repo
  ),
  
  -- oxjob #2112 (9 Nov 2026): PMC's own author-manuscript flag for each article, from PMC's open-data bucket
  -- (openalex.pubmed_central.pmc_oa_articles, #1793; newest version of each article, newest read of it). Keyed like the
  -- PMC repository record (oai:pubmedcentral.nih.gov:<number>). Read by repo_superlocations' version below.
  -- Revert: drop this CTE and the pm join and version case in repo_superlocations.
  pmc_manuscript_flags as (
    select
      concat('oai:pubmedcentral.nih.gov:', regexp_extract(pmcid, '^PMC([0-9]+)$', 1)) as pmh,
      max_by(is_manuscript, struct(version, fetched_at)) as is_manuscript
    from openalex.pubmed_central.pmc_oa_articles
    where error is null and is_manuscript is not null and pmcid rlike '^PMC[0-9]+$'
    group by 1
  ),

  -- oxjob #2066: arXiv's oai_dc records carry no licence, so an arXiv record that got none from its landing page or PDF
  -- takes the licence arXiv registered with DataCite for the same paper (10.48550/arXiv.<id>, client arxiv.content). arXiv's
  -- own default licences (nonexclusive-distrib, assumed-1991-2003) normalise to null there, so only a stated licence is copied.
  -- oxjob #2066 (18 Nov 2026): DataCite rows parsed before normalize_license read the non-BY CC URLs (#314) still hold
  -- null in locations_parsed for ~185K arXiv DOIs (CC BY-NC-SA, BY-NC-ND, BY-SA). datacite_licence_fix (#2178) carries the
  -- licence the current parser reads; it applies under CreateLocationsMapped's own guard (row still null and built
  -- against this updated_date), so the arXiv copy and its DOI location show the same licence (~77K arXiv copies).
  -- Revert: the license column back to p.license, drop the lf join and the coalesce in the where clause.
  arxiv_datacite_licence as (
    select
      concat('oai:arXiv.org:', regexp_extract(lower(p.native_id), '^10\\.48550/arxiv\\.(.+)$', 1)) as pmh,
      coalesce(nullif(p.license, ''), case when lf.fix_updated_date = p.updated_date then lf.fix_license end) as license
    from
      openalex.works.locations_parsed p
      left join openalex.works.datacite_licence_fix lf
        on lf.native_id = p.native_id and lf.native_id_namespace = p.native_id_namespace
    where
      p.provenance = 'datacite'
      and lower(p.native_id) like '10.48550/arxiv.%'
      and coalesce(nullif(p.license, ''), case when lf.fix_updated_date = p.updated_date then lf.fix_license end) is not null
  ),

  -- oxjob #2274 (fix 5; moved above repo_superlocations by oxjob #2542, which reads it there): a DOI with no Crossref record (DataCite theses and data, JaLC and KCI journals carried by a MAG
  -- record) never shows its landing page's affiliations: crossref_super_authorships builds them from the page, but only
  -- a Crossref location reads that table. A DataCite or MAG row with no affiliations takes them, seat for seat, from the
  -- DOI's super-authorships when both list exactly the same author keys in the same order and the DOI's landing page
  -- gives affiliations (so the strings are the page's, not only a PDF parse's); fills empty seats only, and the row's
  -- own author list is kept. Revert: drop these two CTEs and their two LEFT JOINs, and the coalesce/zip edits below.
  --
  -- oxjob #2542 (22 Nov 2026): fixes 5 and 6 matched a record to the page only on exactly the same author keys, and only
  -- DataCite and MAG records. On DOIs with no Crossref record that kept ~107K works from their page's corresponding
  -- author or affiliations: Persian bylines key a compound surname one way on the page and another in MAG ('Najafpour
  -- Qarajeh, Hossein' najafpourqarajeh;h, MAG qarajeh;h), Japanese ones swap given and family name, and a DOAJ
  -- repository or PubMed record often carries the work's byline (Sinaweb, AME). So on DOIs with no Crossref row only
  -- (doi_page_nocr_dois), a DataCite, MAG, PubMed or repository record as long as the page's list, with no affiliation
  -- (or no flag) on any seat, takes the page's values by position when every seat is the same person: the same key, or
  -- one side's key surname (2+ letters) is a whole word of the other side's name (letters of any script; zero-width
  -- marks, apostrophes and '&#039;' dropped; Arabic yeh and kaf and Turkish dotless i folded). Elsewhere fixes 5 and 6
  -- run exactly as before (the first arm of each WHERE). Revert: drop doi_page_nocr_dois, doi_page_carriers and the
  -- second arm of each fill's WHERE (restore the old FROM), and the pubmed_superlocations / repo_superlocations joins.
  doi_page_affiliation_dois as (
    select distinct lower(get(filter(ids, x -> x.namespace = 'doi').id, 0)) as doi
    from openalex.works.locations_parsed
    where provenance = 'landing_page' and affiliations_exist
  ),
  doi_page_nocr_dois as (  -- oxjob #2542: DOIs whose landing page gives affiliations or flags, with no Crossref record
    select p.doi
    from (
      select distinct lower(get(filter(ids, x -> x.namespace = 'doi').id, 0)) as doi
      from openalex.works.locations_parsed
      where provenance = 'landing_page' and (affiliations_exist or is_corresponding_exists)
    ) p
    left anti join (select distinct lower(native_id) as doi from openalex.works.locations_parsed where provenance = 'crossref') cr
      on cr.doi = p.doi
    where p.doi is not null
  ),
  doi_page_carriers as (  -- oxjob #2542: the records the fills may change; a repository record's list as repo_superlocations shows it
    select c.provenance, c.native_id, c.doi, c.affiliations_exist, c.is_corresponding_exists, c.nocr,
      case when c.provenance in ('repo', 'repo_backfill') then coalesce(ra.authorships, c.authors) else c.authors end as authors
    from (
      select o.*, n.doi is not null as nocr
      from (
        select provenance, native_id, authors, affiliations_exist, is_corresponding_exists,
          lower(case when provenance = 'datacite' then native_id else get(filter(ids, x -> x.namespace = 'doi').id, 0) end) as doi
        from openalex.works.locations_parsed
        where provenance in ('datacite', 'mag', 'pubmed', 'repo', 'repo_backfill') and authors_exist
      ) o
      left join doi_page_nocr_dois n on n.doi = o.doi
      where o.doi is not null and (o.provenance in ('datacite', 'mag') or n.doi is not null)
    ) c
    left join openalex.works.repo_super_authorships ra
      on c.provenance in ('repo', 'repo_backfill') and ra.pmh_id = c.native_id
  ),
  doi_page_affiliation_fill as (
    select o.provenance as fill_provenance, o.native_id as fill_native_id, ca.authorships as fill_authorships
    from doi_page_carriers o
    join openalex.works.crossref_super_authorships ca on ca.doi = o.doi
    left semi join doi_page_affiliation_dois pd on pd.doi = o.doi
    where exists(ca.authorships, x -> exists(x.affiliations, f -> length(trim(f.name)) > 0))
      and (
        -- fix 5 (oxjob #2274), unchanged
        (o.provenance in ('datacite', 'mag') and not coalesce(o.affiliations_exist, false)
          and to_json(transform(ca.authorships, x -> x.author_key)) = to_json(transform(o.authors, x -> x.author_key)))
        -- oxjob #2542: no Crossref record, same length, every seat the same person
        or (o.nocr and size(ca.authorships) = size(o.authors)
          and not exists(o.authors, a -> exists(a.affiliations, x -> x.name is not null))
          and forall(zip_with(o.authors, ca.authorships, (a, s) ->
            a.author_key = s.author_key
            or (length(translate(split_part(a.author_key, ';', 1), 'يكı', 'یکi')) >= 2 and array_contains(
              split(trim(regexp_replace(translate(lower(regexp_replace(s.name, '&#0*39;|&apos;|[\\x{200B}-\\x{200F}\\x{00AD}\\x{0027}\\x{2019}\\x{0060}]', '')), 'يكı', 'یکi'), '[^\\p{L}]+', ' ')), ' '),
              translate(split_part(a.author_key, ';', 1), 'يكı', 'یکi')))
            or (length(translate(split_part(s.author_key, ';', 1), 'يكı', 'یکi')) >= 2 and array_contains(
              split(trim(regexp_replace(translate(lower(regexp_replace(a.name, '&#0*39;|&apos;|[\\x{200B}-\\x{200F}\\x{00AD}\\x{0027}\\x{2019}\\x{0060}]', '')), 'يكı', 'یکi'), '[^\\p{L}]+', ' ')), ' '),
              translate(split_part(s.author_key, ';', 1), 'يكı', 'یکi')))), x -> coalesce(x, false)))
      )
    qualify row_number() over (partition by o.provenance, o.native_id order by ca.doi) = 1  -- one row per location
  ),

  -- oxjob #2274 (fix 6): the same DOIs' corresponding author. A DataCite or MAG row that flags no one takes the flags of
  -- the DOI's super-authorships when both list the same author keys in the same order and the DOI's landing page flags
  -- someone without flagging every one of 2+ authors (J-STAGE's '*' and 責任著者 marks). arXiv DOIs are left to #1402's
  -- fill. Revert: drop these two CTEs, their LEFT JOINs and the h. terms below. oxjob #2542: the same second arm as above.
  doi_page_flag_dois as (
    select distinct lower(get(filter(ids, x -> x.namespace = 'doi').id, 0)) as doi
    from openalex.works.locations_parsed
    where provenance = 'landing_page' and is_corresponding_exists
      and exists(authors, a -> coalesce(a.is_corresponding, false))
      and not (size(authors) >= 2 and forall(authors, a -> coalesce(a.is_corresponding, false)))
  ),
  doi_page_flag_fill as (
    select o.provenance as flag_provenance, o.native_id as flag_native_id, ca.authorships as flag_authorships
    from doi_page_carriers o
    join openalex.works.crossref_super_authorships ca on ca.doi = o.doi
    left semi join doi_page_flag_dois pd on pd.doi = o.doi
    where o.doi not like '10.48550/arxiv.%'
      and not exists(o.authors, a -> coalesce(a.is_corresponding, false))
      and exists(ca.authorships, x -> coalesce(x.is_corresponding, false))
      and not (size(ca.authorships) >= 2 and forall(ca.authorships, x -> coalesce(x.is_corresponding, false)))
      and (
        -- fix 6 (oxjob #2274), unchanged
        (o.provenance in ('datacite', 'mag') and not coalesce(o.is_corresponding_exists, false)
          and to_json(transform(ca.authorships, x -> x.author_key)) = to_json(transform(o.authors, x -> x.author_key)))
        -- oxjob #2542: no Crossref record, same length, every seat the same person
        or (o.nocr and size(ca.authorships) = size(o.authors)
          and forall(zip_with(o.authors, ca.authorships, (a, s) ->
            a.author_key = s.author_key
            or (length(translate(split_part(a.author_key, ';', 1), 'يكı', 'یکi')) >= 2 and array_contains(
              split(trim(regexp_replace(translate(lower(regexp_replace(s.name, '&#0*39;|&apos;|[\\x{200B}-\\x{200F}\\x{00AD}\\x{0027}\\x{2019}\\x{0060}]', '')), 'يكı', 'یکi'), '[^\\p{L}]+', ' ')), ' '),
              translate(split_part(a.author_key, ';', 1), 'يكı', 'یکi')))
            or (length(translate(split_part(s.author_key, ';', 1), 'يكı', 'یکi')) >= 2 and array_contains(
              split(trim(regexp_replace(translate(lower(regexp_replace(a.name, '&#0*39;|&apos;|[\\x{200B}-\\x{200F}\\x{00AD}\\x{0027}\\x{2019}\\x{0060}]', '')), 'يكı', 'یکi'), '[^\\p{L}]+', ' ')), ' '),
              translate(split_part(s.author_key, ';', 1), 'يكı', 'یکi')))), x -> coalesce(x, false)))
      )
    qualify row_number() over (partition by o.provenance, o.native_id order by ca.doi) = 1  -- one row per location
  ),

  repo_superlocations as (
    select
      c.provenance,
      c.native_id,
      c.native_id_namespace,
      c.title,
      c.normalized_title,
      case  -- oxjob #2542: the DOI page's affiliations (g) and flags (h) on a record of a DOI with no Crossref record
        when coalesce(g.fill_authorships, h.flag_authorships) is not null
        then zip_with(coalesce(a.authorships, c.authors), coalesce(g.fill_authorships, h.flag_authorships),
          (x, s) -> named_struct('given', x.given, 'family', x.family, 'name', x.name, 'orcid', x.orcid,
            'affiliations', if(exists(x.affiliations, y -> y.name is not null) or g.fill_authorships is null,
                               x.affiliations, s.affiliations),
            'is_corresponding', if(h.flag_authorships is not null, s.is_corresponding, x.is_corresponding),
            'author_key', x.author_key))
        else coalesce(a.authorships, c.authors)
      end as authors,
      c.ids,
      c.raw_type,
      c.type,
      -- oxjob #2112: a PMC copy's version follows PMC's own flag where the open-data bucket has the article: author
      -- manuscript -> acceptedVersion, otherwise publishedVersion, except preprints (bioRxiv, medRxiv, Research Square,
      -- arXiv sets: submittedVersion from the set name, and PMC's flag is false for them). The set-name rule (#1344) read
      -- ~94K manuscripts filed only under a journal set or pmc-open as publishedVersion, and ~6.3K with no set as
      -- submittedVersion. Articles missing from the bucket keep the set-name version.
      case
        when pm.is_manuscript then 'acceptedVersion'
        when not pm.is_manuscript and nullif(o.version, '') is distinct from 'submittedVersion' then 'publishedVersion'
        else nullif(o.version, "")
      end as version,
      coalesce(nullif(o.license, ""), al.license) as license,
      c.language,
      c.published_date,
      c.created_date,
      c.updated_date,
      c.issue,
      c.volume,
      c.first_page,
      c.last_page,
      c.is_retracted,
      COALESCE(o.abstract, c.abstract) as abstract,
      c.source_name,
      c.publisher,
      c.funders,
      o.references,
      c.urls,
      c.mesh,
      -- oxjob #2336: a repository record whose own listed file a fetch confirmed as the work's free full text opens
      -- the copy, as a PDF TaxiCab fetched already does (pdf_urls_for_repo). See repo_free_file_opened's join below.
      o.is_oa or ff.ff_native_id is not null as is_oa,
      coalesce(c.endpoint_id, rw.endpoint_id) as endpoint_id,
      COALESCE(o.abstract_inverted_index, c.abstract_inverted_index) as abstract_inverted_index,
      c.authors_exist,
      c.affiliations_exist or g.fill_authorships is not null as affiliations_exist,  -- oxjob #2542
      c.is_corresponding_exists or h.flag_authorships is not null as is_corresponding_exists,  -- oxjob #2542
      c.best_doi,
      c.merge_key,
      case
        -- oxjob #2102: a withdrawn arXiv paper (arxiv_withdrawals, cell above) has no PDF at its unversioned link (404
        -- 'withdrawn'); the record stays, open at its landing page, which shows the withdrawal (~14.3K locations).
        when aw.pmh is not null then null
        when lower(c.native_id) like '%arxiv.org%' then coalesce(
          concat('https://arxiv.org/pdf/', split_part(c.native_id, ':', 3)),
          p.pdf_url
        )
        -- oxjob #2112: a PMC copy whose chosen crawled page was its own article PDF keeps that PDF here, since the
        -- landing page below becomes the article page.
        when c.pmc_n is not null and p.pdf_url is null
             and l.landing_page_url rlike concat('^https?://pmc\\.ncbi\\.nlm\\.nih\\.gov/articles/PMC', c.pmc_n, '/pdf/[^/?#]+\\.pdf$')
          then l.landing_page_url
        else coalesce(p.pdf_url, ff.ff_pdf_url)  -- oxjob #2336: the confirmed file when TaxiCab has none
      end as pdf_url,
      case
        when lower(c.native_id) like '%arxiv.org%' then coalesce(
          get(filter(c.urls, x -> x.content_type = 'html').url, 0),
          l.landing_page_url
        )
        -- oxjob #2112: a PMC copy's landing page is its own PMC article page. TaxiCab crawls every URL in a PMC
        -- record, and since PMC's new OAI endpoint (Dec 2025) that includes the PubMed abstract page, which won the
        -- newest-crawl tie on 220,050 PMC copies (and their oa_url). Kept as is when it already is the article page
        -- (www.ncbi.nlm.nih.gov/pmc/articles/<n> or pmc.ncbi.nlm.nih.gov/articles/PMC<n>/); anything else (PubMed,
        -- a PDF or supplement file, a publisher page) becomes NCBI's canonical article URL.
        when c.pmc_n is not null
             and coalesce(l.landing_page_url, '') not rlike concat(
               '^https?://(www\\.ncbi\\.nlm\\.nih\\.gov/pmc/articles/(PMC)?', c.pmc_n,
               '|pmc\\.ncbi\\.nlm\\.nih\\.gov/articles/PMC', c.pmc_n, ')/?$')
          then concat('https://pmc.ncbi.nlm.nih.gov/articles/PMC', c.pmc_n, '/')
        else l.landing_page_url
      end as landing_page_url,
      p.pdf_s3_id,
      p.grobid_s3_id,
      c.ingested_at
    from
      (
        select *, nullif(regexp_extract(native_id, '^oai:pubmedcentral\\.nih\\.gov:([0-9]+)$', 1), '') as pmc_n  -- oxjob #2112
        from locations_parsed_checked
      ) c
      left join openalex.repo.repo_works rw on c.native_id = rw.native_id
      left join pdf_urls_for_repo p on c.native_id = p.pmh
      -- oxjob #2336: repository records of closed works whose own listed file served, on a fetch, a PDF of the work's own
      -- text: title words on pages 1-3, more than 2 pages (a thesis 20+), works typed article / preprint / dissertation /
      -- report only (for books, chapters and conference papers a title-matching file was often a library table-of-contents
      -- scan or a slide deck), no abstract / cover / chapter / slides / Digital Commons additional file. Built by oxjobs
      -- #2336 work/crawl/rule.sql from #1220's confirm crawl (set 2336_closed_free_file). Bar, 17 Nov 2026, fresh draw of
      -- the rule's own rows (seed 23361119, Opus 5.5 judge on the file's first 3 pages): 196 of 200 the work's own full
      -- text or a document containing it. Restricted files (401, login) never serve a PDF, so they stay closed.
      -- Revert: drop this join and the two #2336 lines above (is_oa back to o.is_oa, pdf_url back to p.pdf_url).
      -- oxjob #2429: the same join also opens repository records whose own oai_openaire record states open access
      -- (COAR c_abf2) and lists an open file (openalex.repo.repo_stated_open), on feeds whose oai_openaire record can't
      -- replace their oai_dc one (EPrints' carries no DOI; a few move dates or titles): their oai_dc rows stay as they
      -- are and only the access opens. Rows ship only when the record opens a closed work, its file served a PDF to a
      -- fetch, the work is an article / preprint / dissertation / report and the file has more than 2 pages (a thesis
      -- 20+), and the file is no abstract / poster / slides / supplementary / excerpt file (oxjobs #2429
      -- work/stated_open_v4.sql). Bar, 21 Nov 2026, fresh draw (seed 24291125, Opus 5.5 judge on the file's first 3
      -- pages, scans read from the page images): 198 of 200 the work's own full text or a document containing it.
      -- Revert (#2429 only): drop the union's second arm.
      left join (
        -- a fetched-and-checked file (#2336) outranks a stated one (#2429) for the same record, then the newest
        select native_id as ff_native_id, max_by(pdf_url, struct(evidence_rank, checked_at)) as ff_pdf_url
        from (
          select native_id, pdf_url, checked_at, 2 as evidence_rank from openalex.repo.repo_free_file_opened
          union all
          select native_id, pdf_url, checked_at, 1 as evidence_rank from openalex.repo.repo_stated_open
        ) group by native_id
      ) ff on c.native_id = ff.ff_native_id
      left join landing_page_urls_for_repo l on c.native_id = l.pmh
      left join best_other_fields_for_repo o on c.native_id = o.best_pmh
      left join pmc_manuscript_flags pm on c.native_id = pm.pmh  -- oxjob #2112
      left join arxiv_datacite_licence al on c.native_id = al.pmh
      left join openalex.works.arxiv_withdrawals aw on lower(c.native_id) = aw.pmh  -- oxjob #2102 (~17K rows)
      left join openalex.works.repo_super_authorships
      a on c.native_id = a.pmh_id
      left join doi_page_affiliation_fill g on g.fill_provenance = c.provenance and g.fill_native_id = c.native_id  -- oxjob #2542
      left join doi_page_flag_fill h on h.flag_provenance = c.provenance and h.flag_native_id = c.native_id  -- oxjob #2542
      -- oxjob #1647 (Jason + Kyle, 6 Nov 2026): cultural-heritage feeds are no longer harvested. Their records leave here,
      -- listed in openalex.works.oxjob1647_heritage_removed (whole OAI-PMH endpoints, sets, DataCite clients; the ingest
      -- filters keep new ones out), and their works go through the normal delete path. Records of works that are cited or
      -- typed dissertation, conference-paper, book-chapter, review or preprint are not listed: those works stay whole.
      -- Revert: drop this join and the
      -- one in datacite_superlocations, then follow oxjobs #1647 work/heritage/build/REVERT.md. Never drop that table while
      -- the join is live.
      left anti join openalex.works.oxjob1647_heritage_removed hr
        on hr.provenance = c.provenance and hr.native_id = c.native_id
    where
      c.provenance in ('repo', 'repo_backfill')
  ),
  
  -- Use explicit column lists for datacite/mag/pubmed to avoid physical column
  -- order issues with locations_parsed (DLT puts the key column native_id first
  -- in physical storage, but the UNION ALL needs provenance first to match the
  -- explicit CTEs above).
  -- oxjob #1402: arXiv DataCite rows (10.48550/arxiv.*) win the author list for arXiv works but carry
  -- affiliations for only 1-2% of works, while GROBID parses of the same PDFs sit unused on the repo row
  -- (oai:arXiv.org:<id>) or, for older papers, under the DOI. Fill empty affiliations by position from
  -- those super-authorships when the author counts match (same family name at the same position in 99.8%
  -- of seats; Opus 5.5 judged 96.7% of filled seats usable). The DataCite author list itself is kept.
  arxiv_datacite_affiliation_fill as (
    select
      d.native_id,
      case
        when size(ra.authorships) = size(d.authors) and exists(ra.authorships, x -> size(x.affiliations) > 0) then ra.authorships
        when size(ca.authorships) = size(d.authors) and exists(ca.authorships, x -> size(x.affiliations) > 0) then ca.authorships
      end as fill_authorships
    from
      openalex.works.locations_parsed d
      left join openalex.works.repo_super_authorships ra
        on ra.pmh_id = concat('oai:arXiv.org:', regexp_extract(lower(d.native_id), '^10\\.48550/arxiv\\.(.+)$', 1))
      left join openalex.works.crossref_super_authorships ca
        on ca.doi = d.native_id
    where
      d.provenance = 'datacite'
      and lower(d.native_id) like '10.48550/arxiv.%'
      and not coalesce(d.affiliations_exist, false)
  ),

  datacite_superlocations as (
    select
      c.provenance,
      c.native_id,
      c.native_id_namespace,
      c.title,
      c.normalized_title,
      case
        when coalesce(f.fill_authorships, g.fill_authorships, h.flag_authorships) is not null
        then zip_with(c.authors, coalesce(f.fill_authorships, g.fill_authorships, h.flag_authorships),
          (a, s) -> named_struct('given', a.given, 'family', a.family, 'name', a.name, 'orcid', a.orcid,
            'affiliations', if(exists(a.affiliations, x -> x.name is not null)
                               or coalesce(f.fill_authorships, g.fill_authorships) is null, a.affiliations, s.affiliations),
            'is_corresponding', if(h.flag_authorships is not null, s.is_corresponding, a.is_corresponding),  -- oxjob #2274 fix 6
            'author_key', a.author_key))
        else c.authors
      end as authors,
      c.ids,
      c.raw_type,
      c.type,
      c.version,
      c.license,
      c.language,
      c.published_date,
      c.created_date,
      c.updated_date,
      c.issue,
      c.volume,
      c.first_page,
      c.last_page,
      c.is_retracted,
      c.abstract,
      c.source_name,
      c.publisher,
      c.funders,
      c.references,
      c.urls,
      c.mesh,
      c.is_oa,
      c.endpoint_id,
      c.abstract_inverted_index,
      c.authors_exist,
      c.affiliations_exist or coalesce(f.fill_authorships, g.fill_authorships) is not null as affiliations_exist,
      c.is_corresponding_exists or h.flag_authorships is not null as is_corresponding_exists,  -- oxjob #2274 fix 6
      c.best_doi,
      c.merge_key,
      -- oxjob #1646: the DOI's own open PDF. A DataCite location never took its DOI's PDF row; only a Crossref location
      -- joined pdf_urls_for_crossref (keyed by DOI, built from every open pdf row; nothing in it is Crossref's own). So a
      -- DataCite DOI whose PDF OpenAlex had fetched and parsed served no PDF link and no stored PDF or GROBID file
      -- (19 Nov 2026: 1.0M DataCite locations on 978K works; 421K works gain has_content.pdf, 158K a best open copy with a
      -- PDF link). Bar: Opus 5.5 judged 189 of 200 random new links the record's own item (#1646 EXPLORE). Left out: two
      -- prefixes whose DOI serves another item's PDF: 10.34660 (each paper's DOI serves the whole proceedings volume) and
      -- 10.5256 (F1000 data and review DOIs serve the parent article). The page's licence and abstract are NOT taken: they
      -- failed the same bar (licence 132 of 200, site footers; abstract 50 of 200, page boilerplate).
      -- Revert: the three dp. columns back to cast(null as string) and drop the dp join.
      -- oxjob #1646 (21 Nov 2026): a doi.org link is not a PDF link. 386K of these DOI pdf rows carry the DOI itself as
      -- their URL (legacy PDF harvests filed the file under the DOI it was fetched from), so pdf_url repeated the
      -- landing page: 93 of 100 random ones served an HTML page or a bot wall, 4 a PDF. Same rule as #2202 for repository
      -- copies (a pdf_url that is the landing page becomes null). The stored PDF and GROBID file stay (has_content.pdf).
      -- Revert: this CASE back to `dp.pdf_url,`.
      case when dp.pdf_url rlike '^(?i)https?://(dx\\.)?doi\\.org/' then null else dp.pdf_url end as pdf_url,
      concat("https://doi.org/", c.native_id) as landing_page_url,
      dp.pdf_s3_id,
      dp.grobid_s3_id,
      c.ingested_at
    from
      locations_parsed_checked c
      left join pdf_urls_for_crossref dp  -- oxjob #1646
        on dp.doi = c.native_id and not c.native_id rlike '^10\\.(34660|5256)/'
      left join arxiv_datacite_affiliation_fill f on f.native_id = c.native_id
      left join doi_page_affiliation_fill g on g.fill_provenance = 'datacite' and g.fill_native_id = c.native_id  -- oxjob #2274
      left join doi_page_flag_fill h on h.flag_provenance = 'datacite' and h.flag_native_id = c.native_id  -- oxjob #2274 fix 6
      -- oxjob #1647: heritage DataCite clients leave here (see repo_superlocations).
      left anti join openalex.works.oxjob1647_heritage_removed hr
        on hr.provenance = c.provenance and hr.native_id = c.native_id
    where
      c.provenance = 'datacite'
  ),
  
  -- oxjob #2313 / #2571 (Casey 25 Nov 2026: "adjust walden so pdfs can attach to mag"): a PDF fetched from a MAG record's own
  -- link, filed under namespace 'mag-pdf' and the MAG record's native_id, gives that record its reference list and its file ids,
  -- only when the PDF proves it is the record's paper (pdf_for_mag_ok). Its abstract, licence, authors and URL are not used.
  -- Revert: drop pdf_for_mag, pdf_for_mag_cited, pdf_for_mag_ok and the pm join in mag_superlocations.
  pdf_for_mag as (
    select
      get(filter(ids, x -> x.namespace = 'mag-pdf').id, 0) as mag_native_id,
      references as pdf_references,
      get(filter(ids, x -> x.namespace = 'docs.pdf').id, 0) as pdf_file_id,
      get(filter(ids, x -> x.namespace = 'docs.parsed-pdf').id, 0) as grobid_file_id,
      regexp_replace(lower(title), '[^\\p{L}\\p{N}]', '') as pdf_key,
      regexp_replace(lower(concat_ws(' ', title, abstract)), '[^\\p{L}\\p{N}]', '') as head_key,
      regexp_replace(lower(concat_ws(' ', transform(references, r -> concat_ws(' ', r.raw, r.title)))), '[^\\p{L}\\p{N}]', '') as refs_key
    from openalex.works.locations_parsed
    where provenance = 'pdf' and array_contains(ids.namespace, 'mag-pdf')
    qualify row_number() over (partition by get(filter(ids, x -> x.namespace = 'mag-pdf').id, 0)
                               order by size(references) desc nulls last, updated_date desc, native_id) = 1
  ),

  -- The PDF is the record's paper when any of (oxjob #2571, Casey 25 Nov: "go"; pilot 3,053 of 3,737 parsed PDFs, title alone
  -- 2,149; hand checks 10 of 10 right per added class): (a) GROBID's title is the record's (letters and digits, first 150,
  -- similarity >= 0.8); (b) the record's title (15+ letters and digits) opens inside the PDF's title or abstract, or the PDF's
  -- title opens inside the record's; (c) the PDF's reference list prints at least half of MAG's citations of the record (3+;
  -- a cited title's first 40 letters and digits inside the list). (c) is read only for PDFs that fail (a) and (b).
  pdf_for_mag_titled as (
    select pm.*, m.tk,
      coalesce((length(pm.pdf_key) > 0 and length(m.tk) > 0
        and 1 - levenshtein(left(pm.pdf_key, 150), left(m.tk, 150)) / greatest(length(left(pm.pdf_key, 150)), length(left(m.tk, 150))) >= 0.8)
      or (length(m.tk) >= 15 and (instr(pm.head_key, left(m.tk, 60)) > 0
                                  or (length(pm.pdf_key) >= 15 and instr(m.tk, left(pm.pdf_key, 60)) > 0))), false) as title_ok
    from pdf_for_mag pm
    join (select native_id, regexp_replace(lower(title), '[^\\p{L}\\p{N}]', '') as tk from openalex.works.locations_parsed where provenance = 'mag') m
      on m.native_id = pm.mag_native_id
    qualify row_number() over (partition by pm.mag_native_id order by m.tk) = 1
  ),

  pdf_for_mag_cited as (
    select p.mag_native_id, count(*) as n_pairs,
      count_if(length(t.k) >= 15 and instr(p.refs_key, left(t.k, 40)) > 0) as n_printed
    from (select mag_native_id, refs_key from pdf_for_mag_titled where not title_ok and length(refs_key) > 0) p
    join openalex.mid.citation c on c.paper_id = cast(p.mag_native_id as bigint)
    left join openalex.works.merged_work_ids_map mm on mm.id = c.paper_reference_id
    join (select id, regexp_replace(lower(title), '[^\\p{L}\\p{N}]', '') as k from openalex.works.openalex_works) t
      on t.id = coalesce(mm.merge_into_id, c.paper_reference_id)
    group by 1
  ),

  pdf_for_mag_ok as (
    select p.mag_native_id, p.pdf_references, p.pdf_file_id, p.grobid_file_id
    from pdf_for_mag_titled p
    left join pdf_for_mag_cited ci on ci.mag_native_id = p.mag_native_id
    where p.title_ok or (ci.n_pairs >= 3 and ci.n_printed >= 0.5 * ci.n_pairs)
  ),

  mag_superlocations as (
    select
      provenance,
      native_id,
      native_id_namespace,
      title,
      normalized_title,
      case  -- oxjob #2274: the DOI page's affiliations on seats with none (doi_page_affiliation_fill), and its flags (fix 6)
        when coalesce(g.fill_authorships, h.flag_authorships) is not null
        then zip_with(authors, coalesce(g.fill_authorships, h.flag_authorships),
          (a, s) -> named_struct('given', a.given, 'family', a.family, 'name', a.name, 'orcid', a.orcid,
            'affiliations', if(exists(a.affiliations, x -> x.name is not null) or g.fill_authorships is null,
                               a.affiliations, s.affiliations),
            'is_corresponding', if(h.flag_authorships is not null, s.is_corresponding, a.is_corresponding),
            'author_key', a.author_key))
        else authors
      end as authors,
      ids,
      raw_type,
      type,
      version,
      license,
      language,
      published_date,
      created_date,
      updated_date,
      issue,
      volume,
      first_page,
      last_page,
      is_retracted,
      abstract,
      source_name,
      publisher,
      funders,
      case when coalesce(size(references), 0) = 0 and pm.mag_native_id is not null then pm.pdf_references
           else references end as references,  -- oxjob #2313: the record's PDF list (pdf_for_mag_ok)
      urls,
      mesh,
      is_oa,
      endpoint_id,
      abstract_inverted_index,
      authors_exist,
      affiliations_exist or g.fill_authorships is not null as affiliations_exist,  -- oxjob #2274
      is_corresponding_exists or h.flag_authorships is not null as is_corresponding_exists,  -- oxjob #2274 fix 6
      best_doi,
      merge_key,
      cast(null as string) as pdf_url,
      get(urls.url, 0) as landing_page_url,
      pm.pdf_file_id as pdf_s3_id,  -- oxjob #2313 (pdf_for_mag_ok)
      pm.grobid_file_id as grobid_s3_id,
      ingested_at
    from
      openalex.works.locations_parsed lp
      -- oxjob #1647 (6 Nov 2026; worktype charter section 2: library guides are not scholarly). MAG crawled LibGuides
      -- tabs as papers. Their records (frozen list, 1,797,014 rows: MAG has been frozen since 2021, so none arrive anew) are
      -- excluded here, and their works leave through the normal delete path (TrackDeletedWorks). Revert: drop this join
      -- and follow oxjobs #1647 work/libguides/REVERT.md (un-ledger the ids, re-insert the backed-up identity rows).
      left anti join openalex.works.oxjob1647_libguides_removed lg
        on lg.provenance = lp.provenance and lg.native_id = lp.native_id
      left join doi_page_affiliation_fill g on g.fill_provenance = 'mag' and g.fill_native_id = lp.native_id  -- oxjob #2274
      left join doi_page_flag_fill h on h.flag_provenance = 'mag' and h.flag_native_id = lp.native_id  -- oxjob #2274 fix 6
      -- oxjob #2313: this record's own PDF, when it proves it is the record's paper (pdf_for_mag_ok)
      left join pdf_for_mag_ok pm on pm.mag_native_id = lp.native_id
    where
      lp.provenance = 'mag'
  ),
  
  pubmed_superlocations as (
    select
      provenance,
      native_id,
      native_id_namespace,
      title,
      normalized_title,
      case  -- oxjob #2542: the DOI page's affiliations (g) and flags (h) when the DOI has no Crossref record
        when coalesce(g.fill_authorships, h.flag_authorships) is not null
        then zip_with(authors, coalesce(g.fill_authorships, h.flag_authorships),
          (a, s) -> named_struct('given', a.given, 'family', a.family, 'name', a.name, 'orcid', a.orcid,
            'affiliations', if(exists(a.affiliations, x -> x.name is not null) or g.fill_authorships is null,
                               a.affiliations, s.affiliations),
            'is_corresponding', if(h.flag_authorships is not null, s.is_corresponding, a.is_corresponding),
            'author_key', a.author_key))
        else authors
      end as authors,
      ids,
      raw_type,
      type,
      version,
      license,
      language,
      published_date,
      created_date,
      updated_date,
      issue,
      volume,
      first_page,
      last_page,
      is_retracted,
      abstract,
      source_name,
      publisher,
      funders,
      references,
      urls,
      mesh,
      is_oa,
      endpoint_id,
      abstract_inverted_index,
      authors_exist,
      affiliations_exist or g.fill_authorships is not null as affiliations_exist,  -- oxjob #2542
      is_corresponding_exists or h.flag_authorships is not null as is_corresponding_exists,  -- oxjob #2542
      best_doi,
      merge_key,
      cast(null as string) as pdf_url,
      concat("https://pubmed.ncbi.nlm.nih.gov/", native_id) as landing_page_url,
      cast(null as string) as pdf_s3_id,
      cast(null as string) as grobid_s3_id,
      ingested_at
    from
      locations_parsed_checked
      left join doi_page_affiliation_fill g on g.fill_provenance = 'pubmed' and g.fill_native_id = native_id  -- oxjob #2542
      left join doi_page_flag_fill h on h.flag_provenance = 'pubmed' and h.flag_native_id = native_id  -- oxjob #2542
    where
      provenance = 'pubmed'
  ),
  
  superlocations_unioned as (
    select
      *
    from
      crossref_superlocations
    union all
    select
      *
    from
      datacite_superlocations
    union all
    select
      *
    from
      pubmed_superlocations
    union all
    select
      *
    from
      repo_superlocations
    union all
    select
      *
    from
      mag_superlocations
  ),
  
  superlocations_keyed as (
    -- oxjob #733: within one repo endpoint, records exposing an identical URL set
    -- are the same location (OAI re-registrations, dataset firehoses). Batch-only
    -- dedup: streaming layers see historical versions and cannot do this.
    -- Non-repo rows (and repo rows missing endpoint/urls) get a per-row key.
    select
      *,
      case
        when provenance = 'repo'
         and endpoint_id is not null
         and exists(urls, u -> u.url is not null)
        then concat_ws(char(1), 'ep', endpoint_id,
               concat_ws(char(1), array_sort(array_distinct(
                 transform(filter(urls, u -> u.url is not null), u -> trim(u.url))))))
        else concat_ws(char(1), 'row', provenance, native_id, native_id_namespace)
      end as _location_key
    from
      superlocations_unioned
  ),

  superlocations_deduped as (
    select
      * except (_location_key, _rn)
    from (
      select
        *,
        row_number() over (
          partition by _location_key
          order by updated_date desc nulls last, native_id asc
        ) as _rn
      from
        superlocations_keyed
    )
    where
      _rn = 1
  ),

  -- oxjob #733 B1.5 (moved up from works-base; works-base copies strip in Phase 2).
  -- Order matters: the landing fallback must read the normalized urls.
  -- oxjob #2601: an HTML-escaped ampersand in a link is decoded here, for every provenance: '&amp;' (and
  -- '&amp;amp;', '&#38;', '&#x26;') becomes '&'. Feeds that escape their XML twice leave '&amp;' in the stored
  -- link (B3Kat's bvbr.bib-bvb.de:8991/F?func=service&amp;doc_library=..., 3.8M repository records, 24 Nov
  -- 2026), and the server then answers an error page: B3Kat's 'nicht gefunden' on 88 of 88 sampled escaped links,
  -- its record on 88 of 88 decoded ones; of 100 other hosts' links, 20 opened the record only decoded and none only
  -- escaped (a browser following an HTML link sends '&'). Decoded after superlocations_deduped, so the
  -- record-level dedup keys stay as they were, and before superlocations_cleaned, so its punctuation, markup and
  -- item-page repairs read the decoded link (a link ending in '&amp;' no longer looks like one ending in sentence
  -- punctuation). The regex spells the semicolon as \x3B. Revert: the four regexp_replace(..., '&') wrappers
  -- marked oxjob #2601 back to their inner expressions.
  superlocations_final as (
    select
      provenance,
      native_id,
      native_id_namespace,
      title,
      normalized_title,
      authors,
      ids,
      raw_type,
      type,
      version,
      license,
      language,
      published_date,
      created_date,
      updated_date,
      issue,
      volume,
      first_page,
      last_page,
      is_retracted,
      abstract,
      source_name,
      publisher,
      funders,
      references,
      urls,
      mesh,
      is_oa,
      endpoint_id,
      abstract_inverted_index,
      authors_exist,
      affiliations_exist,
      is_corresponding_exists,
      best_doi,
      merge_key,
      regexp_replace(pdf_url, '&(?:(?:(?i:amp)|#0*38|#[xX]0*26)\\x3B)+', '&') as pdf_url,  -- oxjob #2601
      regexp_replace(coalesce(  -- oxjob #2601
        landing_page_url,
        get(filter(urls, x -> x.content_type = 'html').url, 0),
        get(filter(urls, x -> x.content_type = 'pdf').url, 0)
      ), '&(?:(?:(?i:amp)|#0*38|#[xX]0*26)\\x3B)+', '&') as landing_page_url,
      pdf_s3_id,
      grobid_s3_id,
      ingested_at
    from (
      select
        * except (urls),
        transform(
          urls,
          x -> struct(
            regexp_replace(regexp_replace(x.url, 'dx.doi.org', 'doi.org'), '&(?:(?:(?i:amp)|#0*38|#[xX]0*26)\\x3B)+', '&') as url,  -- oxjob #2601
            x.content_type as content_type
          )
        ) as urls
      from
        superlocations_deduped
    )
  ),

  -- oxjob #1220: repository file links a fetch confirmed. TaxiCab's February 2025 bulk harvest stored ~8.4M repository
  -- PDFs under the record page's address and never kept the file's own address, so #2202's rule below leaves those
  -- copies without a pdf_url. openalex.repo.repo_pdf_url_confirmed holds, per copy, the first link we already held (the
  -- record's own urls, the record page's parse, TaxiCab's final address) whose fetch served %PDF (crawl: oxjobs #1220
  -- work/crawl). Only the latest conclusive check counts: a later 'html', 'dead' or 'shell' check (the link stopped
  -- serving a PDF) drops the link; walls and errors (inconclusive) don't. Other candidate sets (#2336) are not read here.
  repo_confirmed_pdf_urls as (
    select native_id as cf_native_id, max_by(pdf_url, checked_at) as cf_pdf_url
    from openalex.repo.repo_pdf_url_confirmed
    where candidate_set = '1220_stored_pdf' and outcome in ('pdf', 'html', 'dead', 'shell')
    group by native_id
    -- figshare links held out (bar on written rows, 17 Nov 2026): a browser opening figshare's /files/<id>.pdf link
    -- lands on the item page, not the PDF; figshare is re-judged on its own.
    having max_by(outcome, checked_at) = 'pdf'
       and not (regexp_extract(max_by(pdf_url, checked_at), '^https?://([^/:]+)', 1) like '%figshare%')
  ),

  -- oxjob #1098 step 3a: OJS installs harvested through their site-wide OAI endpoint (/index.php/index/oai) give each
  -- article's link in the site's "index" context (https://ejournal1.unud.ac.id/index.php/index/article/view/1689, or
  -- /index/article/view/N without index.php). No journal answers there: OJS sends it to its login page, so the copy's
  -- landing page, its fetched page, PDF link and citation all came out empty (767K repository copies, 742K works, 20 Nov
  -- 2026). The record's setSpec is '<journal path>:<section>' ('jupita:ART'). Per OJS record (oai:<host>:article/<n>)
  -- whose own URL list holds such a link: the journal path from its newest harvest, kept only when exactly one.
  repo_ojs_journal_paths as (
    select oj_native_id, get(paths, 0) as oj_journal_path
    from (
      select oj_native_id, max_by(paths, ingested_at) as paths
      from (
        select native_id as oj_native_id, ingested_at,
               array_distinct(transform(filter(set_spec, x -> x rlike '^[A-Za-z0-9_.~-]+:.+$'),
                                        x -> substring_index(x, ':', 1))) as paths
        from openalex.repo.repo_parsed
        where native_id rlike ':article/[0-9]+$' and size(set_spec) > 0
          and exists(urls, u -> u.url rlike '/index/article/view/[0-9]+')
        union all
        select `ns0:header`.`ns0:identifier`, ingested_at,
               array_distinct(transform(filter(`ns0:header`.`ns0:setSpec`, x -> x rlike '^[A-Za-z0-9_.~-]+:.+$'),
                                        x -> substring_index(x, ':', 1)))
        from openalex.repo.repo_reparse_items
        where `ns0:header`.`ns0:identifier` rlike ':article/[0-9]+$' and size(`ns0:header`.`ns0:setSpec`) > 0
          and exists(`ns0:metadata`.`ns1:dc`.`dc:identifier`, x -> x rlike '/index/article/view/[0-9]+')
      )
      group by oj_native_id
    )
    where size(paths) = 1
  ),

  -- oxjob #733 Group-1 (moved up from works-base; works-base copies strip in Phase 2):
  -- junk-license denylist, source_name whitespace/length scrub, and pdf_url backfill
  -- for pdf-only locations (landing IS the record's pdf url -> the pdf exists).
  superlocations_cleaned as (
    select
      provenance,
      native_id,
      native_id_namespace,
      title,
      normalized_title,
      authors,
      ids,
      raw_type,
      type,
      version,
      case
        when license in ('False', 'unspecified-oa', 'gpl', 'gpl-2', 'gpl-3', 'apache-2.0') then null
        else license
      end as license,
      language,
      published_date,
      created_date,
      updated_date,
      issue,
      volume,
      first_page,
      last_page,
      is_retracted,
      abstract,
      case
        when length(trim(regexp_replace(source_name, '\\s+', ' '))) <= 500
          then trim(regexp_replace(source_name, '\\s+', ' '))
      end as source_name,
      publisher,
      funders,
      references,
      urls,
      mesh,
      is_oa,
      endpoint_id,
      abstract_inverted_index,
      authors_exist,
      affiliations_exist,
      is_corresponding_exists,
      best_doi,
      merge_key,
      -- oxjob #2202: a repository copy's pdf_url is a file, never the copy's own landing page. TaxiCab files each PDF it
      -- fetched under the address it started from, which for a repository record is the record page (a handle,
      -- /handle/..., /record/N), so 8.4M repository copies gave their landing page as pdf_url (best open copy of 3.9M
      -- visible works; Unpaywall url_for_pdf). When the PDF link equals the landing page (scheme, www. and trailing slash
      -- ignored) and is not file-shaped, it is null; the landing page, the stored PDF ids and is_oa are untouched. Held:
      -- file-shaped links (.pdf, /bitstream/, /download, /files/ ...) and two hosts whose record address serves the file
      -- (d-nb.info, urlib.net). Bar, 13 Nov 2026: 0 of 300 removed links served a PDF (fresh draw, seed 22022).
      -- Revert: this CASE back to its ELSE branch (`coalesce(pdf_url, case ... end) as pdf_url,`).
      -- oxjob #2601: two file endpoints count as file-shaped. Indico's .../getFile.py/access?... served a PDF on 12 of
      -- 12 fetches (ILC Agenda, 24 Nov 2026), and every KIPF copy whose link is eGovFrame's .../cmm/fms/FileDown.do?...
      -- has a PDF we fetched from that very link (39 of 39). Their links reached this rule once #2601 decoded them.
      -- Revert: drop `|/getfile\\.py/access|/cmm/fms/filedown\\.do` below.
      -- oxjob #1220: where that leaves a repository copy with no pdf_url, the copy takes its confirmed file link
      -- (repo_confirmed_pdf_urls above; every candidate copy is already is_oa, so is_oa does not move). Bar, 14 Nov 2026:
      -- 98 of 100 confirmed links served a PDF on an independent re-fetch, 97 of 98 the work's own file (80 byte-identical
      -- to the PDF we stored), seed 12204. Revert: drop the outer coalesce's second arm and the cf join below.
      coalesce(
      case
        when provenance = 'repo'
             and lower(regexp_replace(coalesce(pdf_url, landing_page_url), '^https?://(www\\.)?|/+$', ''))
                 = lower(regexp_replace(landing_page_url, '^https?://(www\\.)?|/+$', ''))
             and lower(coalesce(pdf_url, landing_page_url)) not rlike
                 'pdf|download|bitstream|/files?/|/content/|viewcontent|/retrieve/|/get/|attachment|/document$|/getfile\\.py/access|/cmm/fms/filedown\\.do|\\.(docx?|pptx?|zip|ps|epub|xml|txt)([?#]|$)'
             and coalesce(pdf_url, landing_page_url) not rlike '^https?://(d-nb\\.info|urlib\\.net)/'
          then null
        else coalesce(
          pdf_url,
          case
            when landing_page_url = get(filter(urls, x -> x.content_type = 'pdf').url, 0)
                 -- oxjob #1935: a record whose pdf-typed link is an image gives no pdf_url (ANU's DSpace thumbnails
                 -- .../bitstream/.../name.pdf.jpg, IssueLab cover .png: ~5.5K repository copies, 17 Nov 2026).
                 -- Revert: drop these three comment lines and the `and not (...)` line below them.
                 and not (landing_page_url rlike '(?i)\\.(jpe?g|png|gif|webp|bmp|tiff?|svg)([?#]|$)' or landing_page_url like '%ars.els-cdn.com/content/image/%')
              then landing_page_url
          end
        )
      end,
        case when provenance = 'repo' then regexp_replace(cf_pdf_url, '&(?:(?:(?i:amp)|#0*38|#[xX]0*26)\\x3B)+', '&') end  -- oxjob #2601
      ) as pdf_url,
      -- oxjob #1453: a repository location's landing page is the item's own page. EPrints puts a whole citation in
      -- dc:identifier ("Smith, J. ORCID: <https://orcid.org/0000-...> (2020) Title. https://doi.org/10.1/x>."), and the
      -- first link in it became the landing page: an ORCID iD, an author or journal browse page (/view/creators/...),
      -- or a link with the citation's markup glued on ('>', '>.', '&quot;&gt;'); other repositories list an OpenURL
      -- resolver query first. When the landing page is one of those, take instead (1) the item's page as the record's
      -- own URL list gives it: for oai:<host>:<n>, the https://<host>/[id/eprint/]<n>/ that its file or item link
      -- starts with; for other ids, a listed link that carries the record's local id (a handle); else (2) the
      -- repository's item-page pattern learned from its other records (openalex.repo.repo_item_url_templates, built by
      -- notebooks/maintenance/BuildRepoItemUrlTemplates.sql); else the landing page stays as it is. Bar, 12 Nov 2026:
      -- 172 of 172 readable fetched pages were the item's own page, and 96% of 875,773 new links appear word for word
      -- in the repository's own raw record. pdf_url above still reads the landing page as it was.
      -- Revert: this CASE back to `landing_page_url,` and drop the tpl join below.
      -- oxjob #1662: a repository landing page that ends in sentence punctuation is a link lifted out of a citation with
      -- the full stop glued on (DSpace puts the whole citation in dc:identifier: '... Open House International.
      -- https://doi.org/10.1108/OHI-02-2024-0065.'), and it 404s (ZD 25939, W4400652809). Same repair as above: the
      -- item's own page from the record's URL list, else the template, else the link without the trailing '.', ',', ';'
      -- or ':' (and a ')' or ']' only when the link opens none). Covers repo_backfill rows too.
      -- Revert: drop the second WHEN-group (the `or (provenance in ...` lines) and the fallback back to `landing_page_url)`.
      -- oxjob #1098 step 3a: an OJS link in the site's "index" context (repo_ojs_journal_paths above) takes the record's
      -- journal path, when the link's article number is the OAI id's: .../index.php/index/article/view/1689 ->
      -- .../index.php/jupita/article/view/1689. Checked 20 Nov 2026 on a fresh 70-host draw: 60 of 60 readable new links were
      -- the record's own article; 0 of 70 old links were an article page (59 ended on the OJS login).
      -- Revert: drop this WHEN, the repo_ojs_journal_paths CTE and its join below.
      case
        when provenance in ('repo', 'repo_backfill') and oj.oj_journal_path is not null
             and regexp_extract(landing_page_url, '/index/article/view/([0-9]+)', 1)
                 = regexp_extract(native_id, ':article/([0-9]+)$', 1)
          then regexp_replace(landing_page_url, '/index/article/view/', concat('/', oj.oj_journal_path, '/article/view/'))
        when (provenance = 'repo' and (
              landing_page_url rlike '^https?://(www\\.)?orcid\\.org(/|$)'
           or landing_page_url rlike '[<>"]|&quot;|&gt;|&lt;'
           or (native_id rlike '^oai:.+:[0-9]+$'
               and landing_page_url rlike '^https?://[^/]+/view/[A-Za-z_]+/[^/]+(\\.html|/)?$'
               and landing_page_url not rlike concat('/', regexp_extract(native_id, '([0-9]+)$', 1), '(/|$)'))
           or lower(landing_page_url) rlike 'url_ver=z39|/openurl([/?.]|$)|/sfx[_/?]|uresolver\\?'))
          or (provenance in ('repo', 'repo_backfill') and (
              landing_page_url rlike '[.,;:]$'
           or (landing_page_url rlike '[)\\]]$' and landing_page_url not rlike '[(\\[]')))
        then coalesce(
          case
            when native_id rlike '^oai:.+:[0-9]+$' then
              get(filter(transform(urls, u -> regexp_extract(
                    regexp_replace(u.url, '([<>"]|&quot;|&gt;|&lt;).*$', ''),
                    concat('^https?://[^/]+/+(id/eprint/)?', regexp_extract(native_id, '([0-9]+)$', 1), '(/|$)'), 0)),
                  u -> u <> ''), 0)
            else
              get(filter(transform(urls, u -> regexp_replace(regexp_replace(u.url, '([<>"]|&quot;|&gt;|&lt;).*$', ''), '[.,;)\\]]+$', '')),
                  u -> length(regexp_extract(native_id, '([^:]+)$', 1)) >= 5
                       and contains(lower(u), lower(regexp_extract(native_id, '([^:]+)$', 1)))
                       and u not rlike '^https?://(www\\.)?orcid\\.org(/|$)'
                       and lower(u) not rlike 'doi\\.org/|url_ver=z39|/openurl([/?.]|$)|/sfx[_/?]|uresolver\\?'
                       and lower(u) not rlike '\\.pdf($|\\?)|/bitstream/'), 0)
          end,
          concat(tpl.item_url_prefix, regexp_extract(native_id, '([0-9]+)$', 1), '/'),
          -- oxjob #1662: no item page found: the same link without its trailing punctuation
          case when landing_page_url rlike '[(\\[]' then regexp_replace(landing_page_url, '[.,;:]+$', '')
               else regexp_replace(landing_page_url, '[.,;:)\\]]+$', '') end)
        else landing_page_url
      end as landing_page_url,
      pdf_s3_id,
      grobid_s3_id,
      ingested_at
    from
      superlocations_final
      -- oxjob #1453: one row per EPrints-style OAI host (a few hundred rows)
      left join openalex.repo.repo_item_url_templates tpl
        on provenance = 'repo' and native_id rlike '^oai:.+:[0-9]+$'
       and tpl.oai_host = regexp_extract(native_id, '^oai:(.+):[0-9]+$', 1)
      -- oxjob #1220: one row per repository copy with a confirmed file link (columns prefixed cf_, so nothing above
      -- becomes ambiguous)
      left join repo_confirmed_pdf_urls cf
        on provenance = 'repo' and cf.cf_native_id = native_id
      -- oxjob #1098 step 3a: one row per OJS record with an index-context link (~811K rows; columns prefixed oj_)
      left join repo_ojs_journal_paths oj
        on provenance in ('repo', 'repo_backfill') and oj.oj_native_id = native_id
  )

  select
    *
  from
    superlocations_cleaned
  where
    -- oxjob #733 location rule: no landing_page_url -> not a location
    landing_page_url is not null
  order by
    provenance ASC,
    native_id ASC,
    updated_date DESC NULLS LAST
)

In [0]:
SELECT * FROM openalex.works.superlocations